# Core-shell viscoelástico: energia interna, limites e vírus da literatura

Notebook ilustrativo do Apêndice I (*Core-shell viscoelástico*) da tese, no mesmo formato do notebook do core-shell fluido. Cada parte apresenta
primeiro as equações (na notação do Apêndice I e dos Caps. 3 a 5) e depois o código que as calcula e as figuras correspondentes. O núcleo
(região $C$, $0\le r<a$, densidade $\rho_C$ e velocidade do som $c_C$) é um fluido não viscoso, a casca ($a\le r\le b$) é um sólido viscoelástico
isotrópico de densidade $\rho_S$ e parâmetros de Lamé complexos $\lambda,\mu$, e o meio externo ($r>b$, $\rho_M$ e $c_M$) é um fluido não viscoso.

1. Configurações e estética das figuras
2. Modelo: potenciais, condições de contorno e coeficientes
3. Energia interna no núcleo e na casca
4. Limites do core-shell viscoelástico
    1. casca sem rigidez ao cisalhamento ($\mu\to0$): core-shell fluido-fluido
    2. casca de espessura nula ($\delta\to0$): esfera fluida
    3. núcleo de raio nulo ($a\to0$): esfera viscoelástica
    4. conservação de energia e comparação com soluções publicadas de cascas elásticas (Fender, 1972; Chang e Demkowicz, 1994)
5. Vírus da literatura: capsídeos como cascas finas
6. Arquivos gerados

**Variável de tamanho.** Como no Apêndice I, $x_M=k_Mb$ é tomado no raio externo, e todas as curvas são de energia interna contra $x_M$.
No capítulo da casca fluida, $x_M=k_Ma$; as funções copiadas daquele notebook são chamadas com $k_Ma=x_M/(1+\delta)$, com $b=(1+\delta)a$.

As curvas de energia interna e a verificação de conservação usam $10^{-4}\leq x_M\leq1$; as figuras dessas curvas têm quatro marcas no eixo horizontal. As comparações com Fender (1972) e Chang e Demkowicz (1994) conservam os intervalos originais das referências, para não comprometer a validação.

Nenhum banco de dados é lido: tudo é calculado aqui.

## 1. Configurações
No Colab, a célula abaixo instala o LaTeX usado nos rótulos (só na primeira vez). No PC, basta ter uma distribuição LaTeX com `dvipng` e `cm-super`.

**Onde ficam os resultados.** No Colab, o Google Drive é montado logo no início, e todas as figuras e tabelas são gravadas direto em
`Meu Drive/coreshell_viscoelastico/resultados/`, de modo que nada se perde quando a sessão termina. Fora do Colab, elas vão para
`resultados/`, na pasta do notebook:

```
resultados/
├── figuras/   limites/ e virus/: png (600 dpi) e pdf de cada figura
└── tabelas/   csv, tex, png, pdf e svg de cada tabela
```

In [ ]:
%%capture
import os, sys, subprocess
if os.path.exists("/content"):
    subprocess.run("apt-get update -qq && apt-get install -y -qq texlive texlive-latex-extra texlive-fonts-recommended dvipng cm-super", shell=True)

In [ ]:
import os, re, glob, json, time, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, NullFormatter, LogLocator, FuncFormatter
from scipy.special import spherical_jn, spherical_yn, eval_legendre
from scipy.optimize import brentq, minimize_scalar
from scipy.signal import find_peaks

USA_LATEX = shutil.which("latex") is not None
RC_TESE = {
    "text.usetex": USA_LATEX,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "text.latex.preamble": r"\usepackage{amsmath}",
    "figure.dpi": 150,                      # só a exibição no notebook; os arquivos saem com OUTPUT_DPI
}
RC_ORIGINAL = {"text.usetex": USA_LATEX, "figure.dpi": 600, "font.family": "sans-serif"}   # rcParams dos notebooks originais do core-shell


def estilo_tese():
    # figuras no estilo do notebook da esfera (LaTeX, Computer Modern)
    global COLORS, LINESTYLES, LINEWIDTHS
    plt.rcdefaults()
    plt.rcParams.update(RC_TESE)
    COLORS, LINESTYLES, LINEWIDTHS = ["#6488A5", "#8B1A1A", "black", "0.45"], ["-", "--", "-.", ":"], [2.0, 2.0, 1.75, 1.6]


def estilo_original():
    # figuras no estilo dos notebooks originais do core-shell (limite homogêneo, sistemas): LaTeX, sans-serif
    plt.rcdefaults()
    plt.rcParams.update(RC_ORIGINAL)


estilo_tese()
VIRG = "{,}" if USA_LATEX else ","          # vírgula decimal ("{,}" evita o espaço extra do LaTeX)

# estética padrão das figuras da tese
OUTPUT_DPI = 1200
COLORS = ["#6488A5", "#8B1A1A", "black", "0.45"]
LINESTYLES = ["-", "--", "-.", ":"]
LINEWIDTHS = [2.0, 2.0, 1.75, 1.6]

# ---------------- pasta dos resultados ----------------
# Google Drive: Meu Drive > coreshell_viscoelastico
try:
    from google.colab import drive
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

if NO_COLAB:
    drive.mount("/content/drive")
    PASTA = "/content/drive/MyDrive/coreshell_viscoelastico"
    SAIDA = os.path.join(PASTA, "resultados")                          # figuras e tabelas, gravadas no Drive
else:
    SAIDA = os.path.abspath(os.environ.get("CORESHELL_SAIDA", "resultados"))
FIG_DIR = os.path.join(SAIDA, "figuras")
TAB_DIR = os.path.join(SAIDA, "tabelas")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)


def dec(valor, casas):
    return f"{valor:.{casas}f}".replace(".", VIRG)


def potencia(v, _=None):
    # rótulo de eixo log em potências de 10
    return "1" if np.isclose(v, 1) else rf"$10^{{{int(round(np.log10(v)))}}}$"


print("LaTeX:", USA_LATEX, "| Colab:", NO_COLAB, "\nfiguras:", FIG_DIR, "\ntabelas:", TAB_DIR)

## 2. Modelo

Uma onda plana monocromática, de potencial $\varphi_{\text{inc}}$, incide ao longo de $+z$ sobre o centro espalhador. Na casca, o deslocamento é
$\mathbf u=-\nabla\psi^{S}+\nabla\times(A^{S}_\phi\hat{\boldsymbol\phi})$ e, com $B=A/(\mathrm i\omega)$, os potenciais em cada região são (Eqs. I.1 a I.4)

$$
\begin{aligned}
\psi^{S}&=B\sum_{\ell}(2\ell+1)\,\mathrm i^\ell\mathcal A_\ell(k_Lr)P_\ell(\cos\theta), &
A^{S}_\phi&=B\sum_{\ell}(2\ell+1)\,\mathrm i^\ell\mathcal B_\ell(k_Tr)\frac{d}{d\theta}P_\ell(\cos\theta),\\
\varphi_M&=A\sum_{\ell}(2\ell+1)\,\mathrm i^\ell\mathcal M_\ell(k_Mr)P_\ell(\cos\theta), &
\varphi_C&=A\sum_{\ell}(2\ell+1)\,\mathrm i^\ell\mathcal C_\ell(k_Cr)P_\ell(\cos\theta),
\end{aligned}
$$

com

$$
\mathcal A_\ell(k_Lr)=a_\ell j_\ell(k_Lr)+\tilde a_\ell y_\ell(k_Lr),\qquad
\mathcal B_\ell(k_Tr)=b_\ell j_\ell(k_Tr)+\tilde b_\ell y_\ell(k_Tr),\qquad
\mathcal M_\ell(k_Mr)=j_\ell(k_Mr)+s_\ell h^{(1)}_\ell(k_Mr),\qquad
\mathcal C_\ell(k_Cr)=c_\ell j_\ell(k_Cr).
$$

Como a origem não pertence à casca, a solução singular $y_\ell$ é mantida nela. As variáveis adimensionais são (Eqs. I.5 e I.27)

$$
x_M=k_Mb,\qquad z_L=k_Lb,\qquad z_T=k_Tb,\qquad x_C=k_Ca,\qquad \zeta_L=k_La,\qquad \zeta_T=k_Ta,
$$

e, com $b=(1+\delta)a$, todas se escrevem em função de $x_M$: $z_L=(c_M/c_L)\,x_M$, $z_T=(c_M/c_T)\,x_M$, $\zeta_L=z_L/(1+\delta)$,
$\zeta_T=z_T/(1+\delta)$ e $x_C=(c_M/c_C)\,x_M/(1+\delta)$, em que $c_L=\sqrt{(\lambda+2\mu)/\rho_S}$ e $c_T=\sqrt{\mu/\rho_S}$ são complexos.

**Condições de contorno.** Em $r=b$, continuidade da velocidade normal, equilíbrio da tração normal e tração tangencial nula (Eqs. I.9, I.13 e I.22):

$$
\begin{aligned}
&x_M\mathcal M'_\ell(x_M)=z_L\mathcal A'_\ell(z_L)+\ell(\ell+1)\mathcal B_\ell(z_T),\\
&z_L^2\big[\lambda\mathcal A_\ell(z_L)-2\mu\mathcal A''_\ell(z_L)\big]-2\mu\,\ell(\ell+1)\big[z_T\mathcal B'_\ell(z_T)-\mathcal B_\ell(z_T)\big]=\rho_Mc_M^2x_M^2\,\mathcal M_\ell(x_M),\\
&2\big[z_L\mathcal A'_\ell(z_L)-\mathcal A_\ell(z_L)\big]+z_T^2\mathcal B''_\ell(z_T)+\big[\ell(\ell+1)-2\big]\mathcal B_\ell(z_T)=0,
\end{aligned}
$$

e as mesmas três condições em $r=a$, com $x_C$, $\zeta_L$, $\zeta_T$, $\rho_Cc_C^2x_C^2\,\mathcal C_\ell(x_C)$ e $\mathcal C'_\ell(x_C)$ no lugar das grandezas externas (Eqs. I.29, I.33 e I.40).
Isolando $s_\ell$ e $c_\ell$, definem-se $A_1,\ldots,\tilde B_6$, $D_1$ e $D_2$ (Eqs. I.11, I.15, I.24, I.25, I.31, I.35, I.42 e I.43), e as seis condições formam o sistema (Eq. I.47)

$$
\begin{pmatrix}
A_1 & \tilde A_1 & B_1 & \tilde B_1 & 0 & -1\\
A_2 & \tilde A_2 & B_2 & \tilde B_2 & 0 & -1\\
A_3 & \tilde A_3 & B_3 & \tilde B_3 & 0 & 0\\
A_4 & \tilde A_4 & B_4 & \tilde B_4 & -1 & 0\\
A_5 & \tilde A_5 & B_5 & \tilde B_5 & -1 & 0\\
A_6 & \tilde A_6 & B_6 & \tilde B_6 & 0 & 0
\end{pmatrix}
\begin{pmatrix}a_\ell\\ \tilde a_\ell\\ b_\ell\\ \tilde b_\ell\\ c_\ell\\ s_\ell\end{pmatrix}
=\begin{pmatrix}-D_1\\ -D_2\\ 0\\ 0\\ 0\\ 0\end{pmatrix}.
$$

Eliminados $s_\ell$ e $c_\ell$, com $A_{12}=A_1-A_2$, $\tilde A_{12}=\tilde A_1-\tilde A_2$, $B_{12}=B_1-B_2$, $\tilde B_{12}=\tilde B_1-\tilde B_2$, $D_{12}=D_2-D_1$ e as
diferenças análogas $A_{45}$, $\tilde A_{45}$, $B_{45}$ e $\tilde B_{45}$ (Eq. I.50), as condições de cisalhamento dão $b_\ell$ e $\tilde b_\ell$ pela regra de Cramer
(Eqs. I.54 a I.58), e o sistema fica $2\times2$ em $a_\ell$ e $\tilde a_\ell$ (Eqs. I.59 a I.65):

$$
\begin{aligned}
\Delta_T&=B_3\tilde B_6-\tilde B_3B_6, &
\widehat\gamma_1&=D_{12}\Delta_T,\\
\widehat\alpha_1&=\Delta_TA_{12}+B_{12}(A_6\tilde B_3-A_3\tilde B_6)-\tilde B_{12}(A_6B_3-A_3B_6), &
\widehat\alpha_2&=\Delta_T\tilde A_{12}+B_{12}(\tilde A_6\tilde B_3-\tilde A_3\tilde B_6)-\tilde B_{12}(\tilde A_6B_3-\tilde A_3B_6),\\
\widehat\beta_1&=\Delta_TA_{45}+B_{45}(A_6\tilde B_3-A_3\tilde B_6)-\tilde B_{45}(A_6B_3-A_3B_6), &
\widehat\beta_2&=\Delta_T\tilde A_{45}+B_{45}(\tilde A_6\tilde B_3-\tilde A_3\tilde B_6)-\tilde B_{45}(\tilde A_6B_3-\tilde A_3B_6),\\
\Delta_L&=\widehat\alpha_1\widehat\beta_2-\widehat\alpha_2\widehat\beta_1, &
a_\ell&=\frac{\widehat\gamma_1\widehat\beta_2}{\Delta_L},\qquad \tilde a_\ell=-\frac{\widehat\gamma_1\widehat\beta_1}{\Delta_L},
\end{aligned}
$$

$$
b_\ell=\frac{(A_6\tilde B_3-A_3\tilde B_6)a_\ell+(\tilde A_6\tilde B_3-\tilde A_3\tilde B_6)\tilde a_\ell}{\Delta_T},\qquad
\tilde b_\ell=-\frac{(A_6B_3-A_3B_6)a_\ell+(\tilde A_6B_3-\tilde A_3B_6)\tilde a_\ell}{\Delta_T},
$$

$$
s_\ell=A_1a_\ell+\tilde A_1\tilde a_\ell+B_1b_\ell+\tilde B_1\tilde b_\ell+D_1,\qquad
c_\ell=A_4a_\ell+\tilde A_4\tilde a_\ell+B_4b_\ell+\tilde B_4\tilde b_\ell\qquad(\text{Eqs. I.66 a I.68}).
$$

Para $\ell=0$, a parcela transversal não existe e as condições de cisalhamento são satisfeitas identicamente; o problema se reduz às quatro
condições de velocidade e de tração normal. No código, isso equivale a tomar $\Delta_T=1$ e $b_0=\tilde b_0=0$, de modo que
$\widehat\alpha_1=A_{12}$, $\widehat\alpha_2=\tilde A_{12}$, $\widehat\beta_1=A_{45}$ e $\widehat\beta_2=\tilde A_{45}$. Os zeros de $\Delta_L$ próximos do eixo real são as ressonâncias.

**Parâmetros da casca.** Os parâmetros de Lamé vêm das velocidades complexas, $\mu=\rho_Sc_T^2$ e $\lambda=\rho_Sc_L^2-2\mu$, ou, para os capsídeos
(Seção 5), de $E$ e $\nu$ com a dissipação só no cisalhamento, como no Cap. 5 (Eqs. 5.83 a 5.85): $\mu=\mu_0(1-\mathrm i\tan\delta_T)$, com
$\mu_0=E/[2(1+\nu)]$, e $K=E/[3(1-2\nu)]$ real, de modo que $\lambda=K-2\mu/3$.

A célula seguinte confere as fórmulas fechadas contra a solução numérica do sistema $6\times6$ da Eq. (I.47), escrito antes das divisões.

In [ ]:
LMAX = 10
RHO_AGUA, C_AGUA = 1024.0, 1522.0          # água, como no meio externo dos capítulos da tese


def J(n, x):
    if n >= 0:
        return spherical_jn(n, x)
    return (-1) ** (-n) * spherical_yn(-n - 1, x)


def Y(n, x):
    if n >= 0:
        return spherical_yn(n, x)
    return (-1) ** (-n - 1) * spherical_jn(-n - 1, x)


def dJ(n, x):
    # f_l' = f_{l-1} - (l+1)/x f_l (vale para qualquer ordem e argumento complexo)
    return J(n - 1, x) - (n + 1) / x * J(n, x)


def dY(n, x):
    return Y(n - 1, x) - (n + 1) / x * Y(n, x)


def H(n, x):
    return J(n, x) + 1j * Y(n, x)


def dH(n, x):
    return dJ(n, x) + 1j * dY(n, x)


def d2J(n, x):
    # segunda derivada pela equação de Bessel esférica: f'' = -(2/x) f' - [1 - l(l+1)/x^2] f
    return -2 / x * dJ(n, x) - (1 - n * (n + 1) / x ** 2) * J(n, x)


def d2Y(n, x):
    return -2 / x * dY(n, x) - (1 - n * (n + 1) / x ** 2) * Y(n, x)


def lame_de_E_nu(E, nu, tan_delta_T=0.0):
    # dissipação só no cisalhamento, K real (Eqs. 5.83 a 5.85): mu = mu_0 (1 - i tan delta_T), lambda = K - 2 mu / 3
    mu0, K = E / (2 * (1 + nu)), E / (3 * (1 - 2 * nu))
    mu = mu0 * (1 - 1j * tan_delta_T)
    return K - 2 * mu / 3, mu


def lame_de_velocidades(rho_S, cL, cT):
    # mu = rho_S c_T^2 e lambda = rho_S c_L^2 - 2 mu (velocidades complexas)
    mu = rho_S * cT ** 2
    return rho_S * cL ** 2 - 2 * mu, mu


def materiais(rho_C, c_C, rho_S, lam, mu, rho_M=RHO_AGUA, c_M=C_AGUA):
    # (rho_M, c_M, rho_C, c_C, rho_S, lambda, mu)
    return rho_M, c_M, rho_C, c_C, rho_S, complex(lam), complex(mu)


def velocidades(mat):
    rM, cM, rC, cC, rS, lam, mu = mat
    return np.sqrt((lam + 2 * mu) / rS), np.sqrt(mu / rS)


def argumentos(xM, delta, mat):
    # (z_L, z_T, zeta_L, zeta_T, x_C), com x_M = k_M b e b = (1 + delta) a
    rM, cM, rC, cC, rS, lam, mu = mat
    cL, cT = velocidades(mat)
    xM = np.asarray(xM, float)
    zL, zT = (cM / cL) * xM, (cM / cT) * xM
    return zL, zT, zL / (1 + delta), zT / (1 + delta), (cM / cC) * xM / (1 + delta)


def coeficientes(ell, xM, delta, mat):
    # a_l, a~_l, b_l, b~_l, c_l, s_l pelas Eqs. (I.11) a (I.68)
    rM, cM, rC, cC, rS, lam, mu = mat
    xM = np.asarray(xM, float)
    zL, zT, zetaL, zetaT, xC = argumentos(xM, delta, mat)
    L = ell * (ell + 1)
    with np.errstate(all="ignore"):
        # superfície externa (r = b) e meio
        jzL, djzL, d2jzL, yzL, dyzL, d2yzL = J(ell, zL), dJ(ell, zL), d2J(ell, zL), Y(ell, zL), dY(ell, zL), d2Y(ell, zL)
        jzT, djzT, d2jzT, yzT, dyzT, d2yzT = J(ell, zT), dJ(ell, zT), d2J(ell, zT), Y(ell, zT), dY(ell, zT), d2Y(ell, zT)
        jM, djM, hM, dhM = J(ell, xM), dJ(ell, xM), H(ell, xM), dH(ell, xM)
        # superfície interna (r = a) e núcleo
        jaL, djaL, d2jaL, yaL, dyaL, d2yaL = J(ell, zetaL), dJ(ell, zetaL), d2J(ell, zetaL), Y(ell, zetaL), dY(ell, zetaL), d2Y(ell, zetaL)
        jaT, djaT, d2jaT, yaT, dyaT, d2yaT = J(ell, zetaT), dJ(ell, zetaT), d2J(ell, zetaT), Y(ell, zetaT), dY(ell, zetaT), d2Y(ell, zetaT)
        jC, djC = J(ell, xC), dJ(ell, xC)
        PM, PC = rM * cM ** 2 * xM ** 2, rC * cC ** 2 * xC ** 2
        # condição cinemática em r = b (Eq. I.11)
        A1, A1t = zL * djzL / (xM * dhM), zL * dyzL / (xM * dhM)
        B1, B1t = L * jzT / (xM * dhM), L * yzT / (xM * dhM)
        D1 = -djM / dhM
        # tração normal em r = b (Eq. I.15)
        A2, A2t = zL ** 2 * (lam * jzL - 2 * mu * d2jzL) / (PM * hM), zL ** 2 * (lam * yzL - 2 * mu * d2yzL) / (PM * hM)
        B2, B2t = -2 * mu * L * (zT * djzT - jzT) / (PM * hM), -2 * mu * L * (zT * dyzT - yzT) / (PM * hM)
        D2 = -jM / hM
        # cisalhamento em r = b (Eqs. I.24 e I.25)
        A3, A3t = 2 * zL * djzL - 2 * jzL, 2 * zL * dyzL - 2 * yzL
        B3, B3t = zT ** 2 * d2jzT + (L - 2) * jzT, zT ** 2 * d2yzT + (L - 2) * yzT
        # condição cinemática em r = a (Eq. I.31)
        A4, A4t = zetaL * djaL / (xC * djC), zetaL * dyaL / (xC * djC)
        B4, B4t = L * jaT / (xC * djC), L * yaT / (xC * djC)
        # tração normal em r = a (Eq. I.35)
        A5, A5t = zetaL ** 2 * (lam * jaL - 2 * mu * d2jaL) / (PC * jC), zetaL ** 2 * (lam * yaL - 2 * mu * d2yaL) / (PC * jC)
        B5, B5t = -2 * mu * L * (zetaT * djaT - jaT) / (PC * jC), -2 * mu * L * (zetaT * dyaT - yaT) / (PC * jC)
        # cisalhamento em r = a (Eqs. I.42 e I.43)
        A6, A6t = 2 * zetaL * djaL - 2 * jaL, 2 * zetaL * dyaL - 2 * yaL
        B6, B6t = zetaT ** 2 * d2jaT + (L - 2) * jaT, zetaT ** 2 * d2yaT + (L - 2) * yaT
        if ell == 0:
            # sem parcela transversal: Delta_T = 1 e b_0 = b~_0 = 0 (restam as quatro condições normais)
            zero = np.zeros_like(zL)
            A3 = A3t = A6 = A6t = B3t = B6 = zero
            B3 = B6t = zero + 1
        # Eq. (I.50)
        A12, A12t, B12, B12t, D12 = A1 - A2, A1t - A2t, B1 - B2, B1t - B2t, D2 - D1
        A45, A45t, B45, B45t = A4 - A5, A4t - A5t, B4 - B5, B4t - B5t
        # Eqs. (I.54), (I.59), (I.61) e (I.64)
        Delta_T = B3 * B6t - B3t * B6
        alpha_1 = Delta_T * A12 + B12 * (A6 * B3t - A3 * B6t) - B12t * (A6 * B3 - A3 * B6)
        alpha_2 = Delta_T * A12t + B12 * (A6t * B3t - A3t * B6t) - B12t * (A6t * B3 - A3t * B6)
        gamma_1 = D12 * Delta_T
        beta_1 = Delta_T * A45 + B45 * (A6 * B3t - A3 * B6t) - B45t * (A6 * B3 - A3 * B6)
        beta_2 = Delta_T * A45t + B45 * (A6t * B3t - A3t * B6t) - B45t * (A6t * B3 - A3t * B6)
        Delta_L = alpha_1 * beta_2 - alpha_2 * beta_1
        # Eqs. (I.65) a (I.68)
        a, at = gamma_1 * beta_2 / Delta_L, -gamma_1 * beta_1 / Delta_L
        b = ((A6 * B3t - A3 * B6t) * a + (A6t * B3t - A3t * B6t) * at) / Delta_T
        bt = -((A6 * B3 - A3 * B6) * a + (A6t * B3 - A3t * B6) * at) / Delta_T
        s = A1 * a + A1t * at + B1 * b + B1t * bt + D1
        c = A4 * a + A4t * at + B4 * b + B4t * bt
    return a, at, b, bt, c, s


def coeficientes_sistema(ell, xM, delta, mat):
    # solução numérica do sistema 6 x 6 (4 x 4 para l = 0), escrito antes das divisões, só para conferência
    rM, cM, rC, cC, rS, lam, mu = mat
    zL, zT, zetaL, zetaT, xC = argumentos(xM, delta, mat)
    L = ell * (ell + 1)
    PM, PC = rM * cM ** 2 * xM ** 2, rC * cC ** 2 * xC ** 2

    def cinematica(z, w):
        return [z * dJ(ell, z), z * dY(ell, z), L * J(ell, w), L * Y(ell, w)]

    def tracao_normal(z, w):
        return [z ** 2 * (lam * J(ell, z) - 2 * mu * d2J(ell, z)), z ** 2 * (lam * Y(ell, z) - 2 * mu * d2Y(ell, z)),
                -2 * mu * L * (w * dJ(ell, w) - J(ell, w)), -2 * mu * L * (w * dY(ell, w) - Y(ell, w))]

    def cisalhamento(z, w):
        return [2 * z * dJ(ell, z) - 2 * J(ell, z), 2 * z * dY(ell, z) - 2 * Y(ell, z),
                w ** 2 * d2J(ell, w) + (L - 2) * J(ell, w), w ** 2 * d2Y(ell, w) + (L - 2) * Y(ell, w)]

    Mx = np.array([cinematica(zL, zT) + [0, -xM * dH(ell, xM)], tracao_normal(zL, zT) + [0, -PM * H(ell, xM)],
                   cisalhamento(zL, zT) + [0, 0], cinematica(zetaL, zetaT) + [-xC * dJ(ell, xC), 0],
                   tracao_normal(zetaL, zetaT) + [-PC * J(ell, xC), 0], cisalhamento(zetaL, zetaT) + [0, 0]], dtype=complex)
    rhs = np.array([xM * dJ(ell, xM), PM * J(ell, xM), 0, 0, 0, 0], dtype=complex)
    if ell == 0:
        linhas, colunas = [0, 1, 3, 4], [0, 1, 4, 5]
        Mx, rhs = Mx[np.ix_(linhas, colunas)], rhs[linhas]
    esc_c, esc_l = np.max(np.abs(Mx), axis=0), np.max(np.abs(Mx), axis=1)     # equilíbrio de escalas (y_l é enorme para argumento pequeno)
    sol = np.linalg.solve(Mx / esc_c / esc_l[:, None], rhs / esc_l) / esc_c
    return np.array([sol[0], sol[1], 0, 0, sol[2], sol[3]]) if ell == 0 else sol


CASOS_TESTE = {   # (núcleo rho_C, c_C; casca rho_S, c_L, c_T; delta)
    "PMMA, perda de 5 graus":       (1024.0, 1522.0, 1190.0, 2700 * np.exp(-5j * np.pi / 180), 1330 * np.exp(-5j * np.pi / 180), 0.3),
    "casca elástica, núcleo de ar": (1.2, 343.0, 1100.0, 1000.0, 400.0, 0.5),
    "k_T real, k_L complexo":       (1024.0, 300.0, 1500.0, 800 * np.exp(-0.02j), 300.0, 1.0),
}
pior = 0.0
for nome, (rC, cC, rS, cL, cT, delta) in CASOS_TESTE.items():
    mat = materiais(rC, cC, rS, *lame_de_velocidades(rS, cL, cT))
    for ell in (0, 1, 2, 5, 9):
        for xM in (0.01, 0.3, 2.0):
            fechada = np.array([complex(v) for v in coeficientes(ell, xM, delta, mat)])
            sistema = coeficientes_sistema(ell, xM, delta, mat)
            ok = np.abs(sistema) > 0
            pior = max(pior, np.max(np.abs(fechada - sistema)[ok] / np.abs(sistema)[ok]))
print(f"maior diferença relativa entre as Eqs. (I.65) a (I.68) e o sistema 6 x 6: {pior:.1e}")

## 3. Energia interna

**Casca.** Em um sólido viscoelástico, a energia média no volume $V$ da casca é (Eqs. 5.13 e 5.27)

$$
W=\frac14\int_V\Big[\rho_S\,\omega^2|\mathbf u|^2+\lambda\,|\nabla\cdot\mathbf u|^2+2\mu\,(\boldsymbol\varepsilon:\boldsymbol\varepsilon^\ast)\Big]dV=\langle T\rangle+U,
$$

e o teorema de Clapeyron harmônico leva a energia armazenada a (Eq. 5.43)

$$
\langle W_{st}\rangle=2\langle T\rangle+\frac14\,\mathrm{Re}\,I_S,\qquad
I_S=\int_{r=b}\sigma_{rr}u_r^\ast\,dS-\int_{r=a}\sigma_{rr}u_r^\ast\,dS,
$$

pois a tração tangencial é nula nas duas faces e a normal exterior em $r=a$ é $-\hat{\mathbf r}$. Com $Z_S=\rho_S|c_L|$, as integrais radiais de
$\langle T\rangle$ reduzem-se a termos de fronteira nas duas superfícies, e, no regime dissipativo ($z_L^2\neq(z_L^\ast)^2$ e $z_T^2\neq(z_T^\ast)^2$),

$$
\begin{aligned}
\frac{2f_0\langle T\rangle}{Z_S|A|^2}={}&\sum_{\ell=0}^{\infty}(2\ell+1)\Bigg\{
|a_\ell|^2\Bigg[\frac{|z_L|^3}{z_L^2-(z_L^\ast)^2}\Big(z_Lj_\ell(z_L)j_\ell'(z_L^\ast)-z_L^\ast j_\ell'(z_L)j_\ell(z_L^\ast)\Big)
-\frac{|\zeta_L|^3}{\zeta_L^2-(\zeta_L^\ast)^2}\Big(\zeta_Lj_\ell(\zeta_L)j_\ell'(\zeta_L^\ast)-\zeta_L^\ast j_\ell'(\zeta_L)j_\ell(\zeta_L^\ast)\Big)\Bigg]\\
&+|\tilde a_\ell|^2\Bigg[\frac{|z_L|^3}{z_L^2-(z_L^\ast)^2}\Big(z_Ly_\ell(z_L)y_\ell'(z_L^\ast)-z_L^\ast y_\ell'(z_L)y_\ell(z_L^\ast)\Big)
-\frac{|\zeta_L|^3}{\zeta_L^2-(\zeta_L^\ast)^2}\Big(\zeta_Ly_\ell(\zeta_L)y_\ell'(\zeta_L^\ast)-\zeta_L^\ast y_\ell'(\zeta_L)y_\ell(\zeta_L^\ast)\Big)\Bigg]\\
&+2\,\mathrm{Re}\Bigg\{a_\ell\tilde a_\ell^\ast\Bigg[\frac{|z_L|^3}{z_L^2-(z_L^\ast)^2}\Big(z_Lj_\ell(z_L)y_\ell'(z_L^\ast)-z_L^\ast j_\ell'(z_L)y_\ell(z_L^\ast)\Big)
-\frac{|\zeta_L|^3}{\zeta_L^2-(\zeta_L^\ast)^2}\Big(\zeta_Lj_\ell(\zeta_L)y_\ell'(\zeta_L^\ast)-\zeta_L^\ast j_\ell'(\zeta_L)y_\ell(\zeta_L^\ast)\Big)\Bigg]\Bigg\}\\
&+\ell(\ell+1)|b_\ell|^2\Bigg[|z_L|j_\ell(z_T)j_\ell(z_T^\ast)-|\zeta_L|j_\ell(\zeta_T)j_\ell(\zeta_T^\ast)
+\frac{|z_T|^2|z_L|}{z_T^2-(z_T^\ast)^2}\Big(z_Tj_\ell(z_T)j_\ell'(z_T^\ast)-z_T^\ast j_\ell'(z_T)j_\ell(z_T^\ast)\Big)
-\frac{|\zeta_T|^2|\zeta_L|}{\zeta_T^2-(\zeta_T^\ast)^2}\Big(\zeta_Tj_\ell(\zeta_T)j_\ell'(\zeta_T^\ast)-\zeta_T^\ast j_\ell'(\zeta_T)j_\ell(\zeta_T^\ast)\Big)\Bigg]\\
&+\ell(\ell+1)|\tilde b_\ell|^2\Bigg[|z_L|y_\ell(z_T)y_\ell(z_T^\ast)-|\zeta_L|y_\ell(\zeta_T)y_\ell(\zeta_T^\ast)
+\frac{|z_T|^2|z_L|}{z_T^2-(z_T^\ast)^2}\Big(z_Ty_\ell(z_T)y_\ell'(z_T^\ast)-z_T^\ast y_\ell'(z_T)y_\ell(z_T^\ast)\Big)
-\frac{|\zeta_T|^2|\zeta_L|}{\zeta_T^2-(\zeta_T^\ast)^2}\Big(\zeta_Ty_\ell(\zeta_T)y_\ell'(\zeta_T^\ast)-\zeta_T^\ast y_\ell'(\zeta_T)y_\ell(\zeta_T^\ast)\Big)\Bigg]\\
&+2\ell(\ell+1)\,\mathrm{Re}\Bigg\{b_\ell\tilde b_\ell^\ast\Bigg[|z_L|j_\ell(z_T)y_\ell(z_T^\ast)-|\zeta_L|j_\ell(\zeta_T)y_\ell(\zeta_T^\ast)
+\frac{|z_T|^2|z_L|}{z_T^2-(z_T^\ast)^2}\Big(z_Tj_\ell(z_T)y_\ell'(z_T^\ast)-z_T^\ast j_\ell'(z_T)y_\ell(z_T^\ast)\Big)
-\frac{|\zeta_T|^2|\zeta_L|}{\zeta_T^2-(\zeta_T^\ast)^2}\Big(\zeta_Tj_\ell(\zeta_T)y_\ell'(\zeta_T^\ast)-\zeta_T^\ast j_\ell'(\zeta_T)y_\ell(\zeta_T^\ast)\Big)\Bigg]\Bigg\}\\
&+2\ell(\ell+1)\,\mathrm{Re}\Bigg\{a_\ell b_\ell^\ast\Big[|z_L|j_\ell(z_L)j_\ell(z_T^\ast)-|\zeta_L|j_\ell(\zeta_L)j_\ell(\zeta_T^\ast)\Big]
+a_\ell\tilde b_\ell^\ast\Big[|z_L|j_\ell(z_L)y_\ell(z_T^\ast)-|\zeta_L|j_\ell(\zeta_L)y_\ell(\zeta_T^\ast)\Big]\\
&\qquad\qquad\qquad+\tilde a_\ell b_\ell^\ast\Big[|z_L|y_\ell(z_L)j_\ell(z_T^\ast)-|\zeta_L|y_\ell(\zeta_L)j_\ell(\zeta_T^\ast)\Big]
+\tilde a_\ell\tilde b_\ell^\ast\Big[|z_L|y_\ell(z_L)y_\ell(z_T^\ast)-|\zeta_L|y_\ell(\zeta_L)y_\ell(\zeta_T^\ast)\Big]\Bigg\}\Bigg\}.
\end{aligned}
$$

Quando $k_L$ é real, o denominador $z_L^2-(z_L^\ast)^2$ se anula, e os colchetes longitudinais passam a ser os das identidades de Lommel para argumento
real, por exemplo $z_L^2j_\ell(z_L)j_\ell'(z_L)+\tfrac{z_L^3}{2}\big[j_\ell^2(z_L)-j_{\ell-1}(z_L)j_{\ell+1}(z_L)\big]$ menos o mesmo termo em $\zeta_L$; quando $k_T$ é real
($q=k_T/|k_L|$ real), os transversais passam a $|z_L|j_\ell^2(z_T)+q|z_L|^2j_\ell(z_T)j_\ell'(z_T)+\tfrac{q^2|z_L|^3}{2}\big[j_\ell^2(z_T)-j_{\ell-1}(z_T)j_{\ell+1}(z_T)\big]$ menos o
mesmo termo na superfície interna, com as trocas $j\to y$ e a forma mista de Lommel para os pares $yy$ e $jy$. O código escolhe a forma em cada caso.

A contribuição de superfície, normalizada da mesma forma, é

$$
\frac{2f_0I_S}{Z_S|A|^2}=-\frac{4}{\rho_S|c_L|^2}\sum_{\ell=0}^{\infty}(2\ell+1)\Bigg\{
\frac{\big[z_L\mathcal A'_\ell(z_L)+\ell(\ell+1)\mathcal B_\ell(z_T)\big]^\ast}{|z_L|}
\Big\{z_L^2\big[\lambda\mathcal A_\ell(z_L)-2\mu\mathcal A''_\ell(z_L)\big]-2\mu\ell(\ell+1)\big[z_T\mathcal B'_\ell(z_T)-\mathcal B_\ell(z_T)\big]\Big\}
-\frac{\big[\zeta_L\mathcal A'_\ell(\zeta_L)+\ell(\ell+1)\mathcal B_\ell(\zeta_T)\big]^\ast}{|\zeta_L|}
\Big\{\zeta_L^2\big[\lambda\mathcal A_\ell(\zeta_L)-2\mu\mathcal A''_\ell(\zeta_L)\big]-2\mu\ell(\ell+1)\big[\zeta_T\mathcal B'_\ell(\zeta_T)-\mathcal B_\ell(\zeta_T)\big]\Big\}\Bigg\},
$$

e $\dfrac{2f_0\langle W_{st}\rangle}{Z_S|A|^2}=2\,\dfrac{2f_0\langle T\rangle}{Z_S|A|^2}+\dfrac14\,\mathrm{Re}\,\dfrac{2f_0I_S}{Z_S|A|^2}$.

**Núcleo.** O potencial tem a forma da esfera fluida, com $c_\ell$ no lugar do coeficiente interno, e a energia é a da Eq. (4.8) com $x_C$:

$$
\frac{2f_0W_T^{(C)}}{Z_C|A|^2}=\sum_{\ell=0}^{\infty}|c_\ell|^2(2\ell+1)\Big\{x_C^3\big[j_\ell^2(x_C)-j_{\ell-1}(x_C)j_{\ell+1}(x_C)\big]+x_C^2\,j_\ell(x_C)\,j_\ell'(x_C)\Big\}.
$$

**Energia total.** Cada região sai normalizada pela própria impedância; para somar com a mesma referência ($Z_M$), como no core-shell fluido,

$$
\widetilde W_T=\frac{Z_C}{Z_M}\,\frac{2f_0W_T^{(C)}}{Z_C|A|^2}+\frac{Z_S}{Z_M}\,\frac{2f_0\langle W_{st}\rangle}{Z_S|A|^2}.
$$

Só a soma em $\ell$ precisa ser truncada; adota-se $\ell_{\max}=10$, o mesmo do core-shell fluido. A célula seguinte confere a energia da casca contra a
integração numérica direta da Eq. (5.13) no volume $a\le r\le b$, por quadratura de Gauss-Legendre em $(r,\theta)$, com o tensor de deformações montado
componente a componente, como na Fig. 23 do Cap. 5. Os casos de teste cobrem as três formas das reduções: $k_L$ e $k_T$ complexos, os dois reais, e
$k_T$ real com $k_L$ complexo.

In [ ]:
TOL_REAL = 1e-9          # |Im z| / |z| abaixo disso: número de onda real (formas de Lommel)
DERIVADA = {J: dJ, Y: dY}


def prim_F(ell, x, f, g):
    # x^3 [f_l g_l - (f_{l-1} g_{l+1} + f_{l+1} g_{l-1})/2]: com f = g = j dá x^3 (j_l^2 - j_{l-1} j_{l+1})
    return x ** 3 * (f(ell, x) * g(ell, x) - 0.5 * (f(ell - 1, x) * g(ell + 1, x) + f(ell + 1, x) * g(ell - 1, x)))


def prim_B(ell, x, f, g):
    # termo de superfície x^2 f_l g_l'
    return x ** 2 * f(ell, x) * (g(ell - 1, x) - (ell + 1) / x * g(ell, x))


def eh_real(z):
    return bool(np.all(np.abs(np.imag(z)) <= TOL_REAL * np.abs(z)))


def fronteira_longitudinal(ell, z, f, g):
    # I_1^{fg} + l(l+1) I_4^{fg} em uma superfície de argumento longitudinal z (z_L ou zeta_L)
    df, dg = DERIVADA[f], DERIVADA[g]
    if eh_real(z):                                   # k_L real: Lommel
        z = np.real(z)
        return z ** 2 * df(ell, z) * g(ell, z) + prim_F(ell, z, f, g) / 2
    zc = np.conj(z)
    return np.abs(z) ** 3 / (z ** 2 - zc ** 2) * (z * f(ell, z) * dg(ell, zc) - zc * df(ell, z) * g(ell, zc))


def fronteira_transversal(ell, zl, w, f, g):
    # I_2 + |eta|^2 I_5 + q* I_8^{fg} + q (I_8^{gf})* + l(l+1) I_2 em uma superfície (|z_L| = |zl|, argumento transversal w)
    df, dg = DERIVADA[f], DERIVADA[g]
    m = np.abs(zl)
    if eh_real(w):                                   # k_T real: q = w/|zl| real e Lommel
        w = np.real(w)
        return (m * f(ell, w) * g(ell, w) + w * m / 2 * (f(ell, w) * dg(ell, w) + df(ell, w) * g(ell, w))
                + m / (2 * w) * prim_F(ell, w, f, g))
    wc = np.conj(w)
    return m * f(ell, w) * g(ell, wc) + np.abs(w) ** 2 * m / (w ** 2 - wc ** 2) * (w * f(ell, w) * dg(ell, wc) - wc * df(ell, w) * g(ell, wc))


def fronteira_acoplamento(ell, zl, w, f, g):
    # e^{i phi_L} I_3^{fg} + I_6^{fg} + q* I_7^{fg} em uma superfície: |z_L| f(z_L) g(z_T*)
    return np.abs(zl) * f(ell, zl) * g(ell, np.conj(w))


def deslocamento_radial(ell, z, w, a, at, b, bt):
    # z A_l'(z) + l(l+1) B_l(w)
    return z * (a * dJ(ell, z) + at * dY(ell, z)) + ell * (ell + 1) * (b * J(ell, w) + bt * Y(ell, w))


def tracao_radial(ell, z, w, a, at, b, bt, lam, mu):
    # z^2 [lambda A_l(z) - 2 mu A_l''(z)] - 2 mu l(l+1) [w B_l'(w) - B_l(w)]
    return (z ** 2 * (a * (lam * J(ell, z) - 2 * mu * d2J(ell, z)) + at * (lam * Y(ell, z) - 2 * mu * d2Y(ell, z)))
            - 2 * mu * ell * (ell + 1) * (b * (w * dJ(ell, w) - J(ell, w)) + bt * (w * dY(ell, w) - Y(ell, w))))


def energia_modal(ell, xM, delta, mat):
    # contribuição do multipolo l: (W~_T, W~_C, W~_S); W~_C normalizada por Z_C, W~_S por Z_S = rho_S |c_L| e W~_T por Z_M
    rM, cM, rC, cC, rS, lam, mu = mat
    cL, cT = velocidades(mat)
    xM = np.asarray(xM, float)
    zL, zT, zetaL, zetaT, xC = argumentos(xM, delta, mat)
    L, p = ell * (ell + 1), 2 * ell + 1
    with np.errstate(all="ignore"):
        a, at, b, bt, c, s = coeficientes(ell, xM, delta, mat)
        WC = np.real(np.abs(c) ** 2 * p * (prim_F(ell, xC, J, J) + prim_B(ell, xC, J, J)))          # W_P + W_K do núcleo
        GL = {fg: fronteira_longitudinal(ell, zL, *fg) - fronteira_longitudinal(ell, zetaL, *fg) for fg in ((J, J), (Y, Y), (J, Y))}
        GT = {fg: fronteira_transversal(ell, zL, zT, *fg) - fronteira_transversal(ell, zetaL, zetaT, *fg) for fg in ((J, J), (Y, Y), (J, Y))}
        GX = {fg: fronteira_acoplamento(ell, zL, zT, *fg) - fronteira_acoplamento(ell, zetaL, zetaT, *fg)
              for fg in ((J, J), (J, Y), (Y, J), (Y, Y))}
        T = p * np.real(np.abs(a) ** 2 * GL[J, J] + np.abs(at) ** 2 * GL[Y, Y] + 2 * np.real(a * np.conj(at) * GL[J, Y])
                        + L * (np.abs(b) ** 2 * GT[J, J] + np.abs(bt) ** 2 * GT[Y, Y] + 2 * np.real(b * np.conj(bt) * GT[J, Y]))
                        + 2 * L * np.real(a * np.conj(b) * GX[J, J] + a * np.conj(bt) * GX[J, Y]
                                          + at * np.conj(b) * GX[Y, J] + at * np.conj(bt) * GX[Y, Y]))
        sup_b = np.conj(deslocamento_radial(ell, zL, zT, a, at, b, bt)) * tracao_radial(ell, zL, zT, a, at, b, bt, lam, mu) / np.abs(zL)
        sup_a = (np.conj(deslocamento_radial(ell, zetaL, zetaT, a, at, b, bt))
                 * tracao_radial(ell, zetaL, zetaT, a, at, b, bt, lam, mu) / np.abs(zetaL))
        IS = -4 / (rS * np.abs(cL) ** 2) * p * (sup_b - sup_a)
        WS = 2 * T + np.real(IS) / 4                                                                   # Eq. (5.43) na casca
        WT = rC * cC / (rM * cM) * WC + rS * np.abs(cL) / (rM * cM) * WS
    return [np.where(np.isfinite(w), w, np.nan) for w in (WT, WC, WS)]


def digitos_perdidos(xM, delta, mat):
    # j_l e y_l crescem como e^{|Im zeta|} enquanto a combinação física decai através da casca: perda de ~2|Im zeta|/ln 10 algarismos
    zL, zT, zetaL, zetaT, xC = argumentos(np.max(xM), delta, mat)
    return 2 * max(abs(np.imag(zetaL)), abs(np.imag(zetaT))) / np.log(10)


def energia_total(xM, delta, mat, lmax=LMAX, partes=False):
    # W~_T = soma de l = 0 a lmax; com partes=True devolve também núcleo e casca, ambos já em relação a Z_M
    rM, cM, rC, cC, rS, lam, mu = mat
    cL, _ = velocidades(mat)
    xM = np.asarray(xM, float)
    if digitos_perdidos(xM, delta, mat) > 6:
        warnings.warn(f"|Im zeta| grande: cerca de {digitos_perdidos(xM, delta, mat):.0f} algarismos perdidos na base j_l, y_l da casca")
    W, WC, WS = np.zeros_like(xM), np.zeros_like(xM), np.zeros_like(xM)
    for ell in range(lmax + 1):
        w = energia_modal(ell, xM, delta, mat)
        W += np.nan_to_num(w[0])
        WC += np.nan_to_num(w[1])
        WS += np.nan_to_num(w[2])
    if partes:
        return W, rC * cC / (rM * cM) * WC, rS * np.abs(cL) / (rM * cM) * WS
    return W


# ---------------- conferência: integração direta da Eq. (5.13) no volume da casca ----------------
def legendre_theta(ell, th):
    # P_l(cos th), dP_l/dth e d^2P_l/dth^2 (esta pela equação de Legendre)
    x = np.cos(th)
    P = eval_legendre(ell, x)
    if ell == 0:
        return P, 0 * th, 0 * th
    dP = -np.sin(th) * ell * (x * P - eval_legendre(ell - 1, x)) / (x ** 2 - 1)
    return P, dP, -ell * (ell + 1) * P - np.cos(th) / np.sin(th) * dP


def energia_casca_volume(xM, delta, mat, lmax=LMAX, nr=140, nt=140):
    # 2 f_0 <W_st> / (Z_S |A|^2) por quadratura em (r, theta), com r em unidades de b; devolve também o erro de tr(eps) contra k_L^2 psi
    rM, cM, rC, cC, rS, lam, mu = mat
    cL, cT = velocidades(mat)
    zL, zT = (complex(v) for v in argumentos(xM, delta, mat)[:2])
    r0 = 1 / (1 + delta)
    gr, wr = np.polynomial.legendre.leggauss(nr)
    gt, wt = np.polynomial.legendre.leggauss(nt)
    rr, wr = r0 + (1 - r0) * (gr + 1) / 2, wr * (1 - r0) / 2
    th, wt = np.pi * (gt + 1) / 2, wt * np.pi / 2
    R, TH = np.meshgrid(rr, th, indexing="ij")
    Ur, Ut, dUr_r, dUt_r, dUr_t, dUt_t, tr_psi = (np.zeros(R.shape, complex) for _ in range(7))
    for ell in range(lmax + 1):
        a, at, b, bt, c, s = (complex(v) for v in coeficientes(ell, xM, delta, mat))
        L, fat = ell * (ell + 1), (2 * ell + 1) * 1j ** ell
        P, dP, d2P = legendre_theta(ell, TH)
        x, w = zL * R, zT * R
        A_, dA, d2A = a * J(ell, x) + at * Y(ell, x), a * dJ(ell, x) + at * dY(ell, x), a * d2J(ell, x) + at * d2Y(ell, x)
        B_, dB, d2B = b * J(ell, w) + bt * Y(ell, w), b * dJ(ell, w) + bt * dY(ell, w), b * d2J(ell, w) + bt * d2Y(ell, w)
        radial, angular = zL * dA + L / R * B_, A_ / R + zT * dB + B_ / R
        Ur += fat * radial * P
        Ut += fat * angular * dP
        dUr_r += fat * (zL ** 2 * d2A + L * (zT * dB / R - B_ / R ** 2)) * P
        dUt_r += fat * (zL * dA / R - A_ / R ** 2 + zT ** 2 * d2B + zT * dB / R - B_ / R ** 2) * dP
        dUr_t += fat * radial * dP
        dUt_t += fat * angular * d2P
        tr_psi += fat * (-zL ** 2) * A_ * P
    e_rr, e_tt = dUr_r, (dUt_t + Ur) / R
    e_pp, e_rt = (Ur + Ut / np.tan(TH)) / R, 0.5 * (dUr_t / R + dUt_r - Ut / R)
    tr = e_rr + e_tt + e_pp
    sigma_eps = lam * np.abs(tr) ** 2 + 2 * mu * (np.abs(e_rr) ** 2 + np.abs(e_tt) ** 2 + np.abs(e_pp) ** 2 + 2 * np.abs(e_rt) ** 2)
    peso = wr[:, None] * wt[None, :] * R ** 2 * np.sin(TH) * 2 * np.pi
    T = np.abs(zL) / (4 * np.pi) * np.sum(peso * (np.abs(Ur) ** 2 + np.abs(Ut) ** 2))
    U = np.sum(peso * sigma_eps) / (4 * np.pi * rS * np.abs(cL) ** 2 * np.abs(zL))
    return T + np.real(U), np.max(np.abs(tr - tr_psi)) / np.max(np.abs(tr_psi))


print(f"{'caso':30s} {'x_M':>5s}   {'forma fechada':>16s}   {'volume':>16s}   {'dif. rel.':>9s}   tr(eps) x k_L^2 psi")
for nome, (rC, cC, rS, cL, cT, delta) in CASOS_TESTE.items():
    mat = materiais(rC, cC, rS, *lame_de_velocidades(rS, cL, cT))
    for xM in (0.3, 2.0):
        Wf = sum(energia_modal(l, xM, delta, mat)[2] for l in range(LMAX + 1))
        Wv, err_tr = energia_casca_volume(xM, delta, mat)
        print(f"{nome:30s} {xM:5.2f}   {Wf:16.10e}   {Wv:16.10e}   {abs(Wf - Wv) / abs(Wv):9.1e}   {err_tr:.1e}")

## 4. Limites do core-shell viscoelástico

A formulação precisa recuperar os três sistemas já tratados na tese quando uma das regiões perde a sua identidade. As funções de referência são as
dos capítulos anteriores, copiadas sem alteração na célula seguinte, e a energia é sempre normalizada por $Z_M$.

1. **Casca sem rigidez ao cisalhamento** ($\mu\to0$ com $K$ fixo). Então $c_T\to0$, $z_T\to\infty$ e, pelo mesmo argumento das Eqs. (5.10) e (5.11),
$b_\ell,\tilde b_\ell\to0$, enquanto $\lambda\to K$. A casca passa a ser um fluido de densidade $\rho_S$ e velocidade $c_B=\sqrt{K/\rho_S}$, e o sistema volta ao
core-shell fluido do Cap. 4 (Eqs. 4.5), com $a_\ell\to c_\ell^{(\mathrm{fl})}$, $\tilde a_\ell\to d_\ell^{(\mathrm{fl})}$, $c_\ell\to b_\ell^{(\mathrm{fl})}$ e $s_\ell\to s_\ell^{(\mathrm{fl})}$, calculados em
$k_Ma=x_M/(1+\delta)$. O limite é tomado ao longo de uma sequência de $c_{T,0}/c_B$ decrescente, com $\mu=\mu_0(1-\mathrm i\tan\delta_T)$ e uma perda pequena,
$\tan\delta_T=0{,}02$, que amortece as ressonâncias estreitas de cisalhamento da casca, sem mudar o limite.
2. **Casca de espessura nula** ($\delta\to0$). Então $a\to b$, o volume da casca vai a zero e a partícula é o núcleo $C$ de raio $b$ no meio $M$: a esfera
fluida do Cap. 3, com $c_\ell\to b^{\mathrm{esf}}_\ell$ e $s_\ell\to s^{\mathrm{esf}}_\ell$. A energia da casca vai a zero; quando a casca é muito mais rígida que o núcleo,
como aqui, seu efeito persiste até espessuras bem pequenas.
3. **Núcleo de raio nulo** ($a\to0$, isto é, $\delta\to\infty$ com $b$ fixo). Então $\zeta_L,\zeta_T\to0$, a regularidade na origem elimina as soluções singulares
($\tilde a_\ell,\tilde b_\ell\to0$) e as parcelas avaliadas em $\zeta_L$ e $\zeta_T$ desaparecem, como discutido no Apêndice I: a partícula é a esfera viscoelástica do
Cap. 5, com $a_\ell$, $b_\ell$ e $s_\ell$ das Eqs. (5.7) a (5.9) e a energia da Eq. (5.58). A energia do núcleo vai a zero.

Nos dois primeiros limites, a convergência dos coeficientes não é uniforme em $\ell$: para $\ell$ alto e $x_M$ pequeno, o coeficiente de espalhamento é
minúsculo e a casca ainda pesa nele, mas a diferença cai como $(c_{T,0}/c_B)^2$ no primeiro limite e como $\delta$ no segundo, como mostra a célula seguinte.

**Observação numérica.** Com perda no cisalhamento e $|\zeta_T|$ grande, $j_\ell$ e $y_\ell$ crescem como $e^{|\mathrm{Im}\,\zeta_T|}$, enquanto a combinação física
decai através da casca; a base $j_\ell,y_\ell$ perde cerca de $2|\mathrm{Im}\,\zeta_T|/\ln10$ algarismos, e `energia_total` avisa quando a perda passa de seis.
É por isso que o limite $\mu\to0$ usa uma perda pequena.

In [ ]:
# ---------------- core-shell fluido (Cap. 4): funções do notebook do core-shell fluido, com o sufixo _fluido; x_M^(fl) = k_M a
def argumentos_fluido(xM, delta, mat):
    rM, rA, rB, cM, cA, cB = mat
    xM = np.asarray(xM)
    xB = (cM / cB) * xM
    return (cM / cA) * xM, xB, (1 + delta) * xM, (1 + delta) * xB


def gamma_contorno(f, df, g, dg, rho_mu_nu, c_nu_mu):
    # Gamma^{fg}_{mu nu} = (rho_mu/rho_nu) f(s_mu) g'(s_nu) - (c_nu/c_mu) f'(s_mu) g(s_nu)   (Eq. 4.4)
    return rho_mu_nu * f * dg - c_nu_mu * df * g


def coeficientes_fluido(ell, xM, delta, mat):
    # b_l, c_l, d_l, s_l e D_l pelas Eqs. (4.5a)-(4.5e)
    rM, rA, rB, cM, cA, cB = mat
    xA, xB, yM, yB = argumentos_fluido(xM, delta, mat)
    with np.errstate(all="ignore"):
        jA, djA = J(ell, xA), dJ(ell, xA)
        jB, djB, yxB, dyxB = J(ell, xB), dJ(ell, xB), Y(ell, xB), dY(ell, xB)
        jb, djb, yb, dyb = J(ell, yB), dJ(ell, yB), Y(ell, yB), dY(ell, yB)
        jM, djM, hM, dhM = J(ell, yM), dJ(ell, yM), H(ell, yM), dH(ell, yM)
        G_jj_AB = gamma_contorno(jA, djA, jB, djB, rA / rB, cB / cA)
        G_jy_AB = gamma_contorno(jA, djA, yxB, dyxB, rA / rB, cB / cA)
        G_hj_MB = gamma_contorno(hM, dhM, jb, djb, rM / rB, cB / cM)
        G_hy_MB = gamma_contorno(hM, dhM, yb, dyb, rM / rB, cB / cM)
        G_jj_MB = gamma_contorno(jM, djM, jb, djb, rM / rB, cB / cM)
        G_jy_MB = gamma_contorno(jM, djM, yb, dyb, rM / rB, cB / cM)
        D = G_jy_AB * G_hj_MB - G_jj_AB * G_hy_MB
        pre = -1j / yM ** 2 * (rM / rB) * (cB / cM)
        return pre / (D * xB ** 2), pre * G_jy_AB / D, -pre * G_jj_AB / D, (G_jj_AB * G_jy_MB - G_jy_AB * G_jj_MB) / D, D


def energia_modal_fluido(ell, xM, delta, mat):
    # contribuição do multipolo l: (W~_T, W~_A, W~_B), normalizadas por Z_M (W~_T = alpha W~_A + beta W~_B)
    rM, rA, rB, cM, cA, cB = mat
    xA, xB, yM, yB = argumentos_fluido(np.asarray(xM, float), delta, mat)
    with np.errstate(all="ignore"):
        b, c, d, _, _ = coeficientes_fluido(ell, xM, delta, mat)
        p = 2 * ell + 1
        FA = prim_F(ell, xA, J, J) / 2
        WA = np.real(np.abs(b) ** 2 * p * (2 * FA + prim_B(ell, xA, J, J)))                      # W_P + W_K do núcleo
        Fjj, Fyy, Fjy = (prim_F(ell, yB, f, g) - prim_F(ell, xB, f, g) for f, g in ((J, J), (Y, Y), (J, Y)))
        Bjj, Byy, Bjy = (prim_B(ell, yB, f, g) - prim_B(ell, xB, f, g) for f, g in ((J, J), (Y, Y), (J, Y)))
        c2, d2, cd = np.abs(c) ** 2, np.abs(d) ** 2, np.real(np.conj(c) * d)
        WP = p * (0.5 * c2 * Fjj + 0.5 * d2 * Fyy + cd * Fjy)
        WB = np.real(2 * WP + p * (c2 * Bjj + d2 * Byy + 2 * cd * Bjy))                           # W_P + W_K da casca
        alpha, beta = rA * cA / (rM * cM), rB * cB / (rM * cM)
        WT = alpha * WA + beta * WB
    return [np.where(np.isfinite(w), w, np.nan) for w in (WT, WA, WB)]


def energia_total_fluido(xM, delta, mat, lmax=LMAX):
    # W~_T = soma de l = 0 a lmax
    xM = np.asarray(xM, float)
    W = np.zeros_like(xM)
    for ell in range(lmax + 1):
        W += np.nan_to_num(energia_modal_fluido(ell, xM, delta, mat)[0])
    return W


# ---------------- esfera fluida (Cap. 3), x = k_M R
def esfera(ell, x, mat_esfera):
    # coeficientes da esfera homogênea S de raio R em M, com x = k_M R; mat_esfera = (rho_M, rho_S, c_M, c_S)
    rM, rS, cM, cS = mat_esfera
    xS, mt = (cM / cS) * x, (rM * cM) / (rS * cS)
    with np.errstate(all="ignore"):
        D = dH(ell, x) * J(ell, xS) - mt * dJ(ell, xS) * H(ell, x)
        return 1j / x ** 2 * (rM / rS) / D, -(dJ(ell, x) * J(ell, xS) - mt * J(ell, x) * dJ(ell, xS)) / D


def energia_esfera(x, mat_esfera, lmax=LMAX):
    # W~_T da esfera homogênea normalizada por Z_M
    rM, rS, cM, cS = mat_esfera
    xS, W = (cM / cS) * np.asarray(x, float), 0.0
    for ell in range(lmax + 1):
        b, _ = esfera(ell, x, mat_esfera)
        W = W + np.real(np.abs(b) ** 2 * (2 * ell + 1) * (prim_F(ell, xS, J, J) + prim_B(ell, xS, J, J)))
    return (rS * cS) / (rM * cM) * W


# ---------------- esfera viscoelástica (Cap. 5), x_M = k_M R
def esfera_viscoelastica(ell, xM, mat_esfera):
    # a_l, b_l, s_l pelas Eqs. (5.7) a (5.9); mat_esfera = (rho_M, c_M, rho_S, lambda, mu)
    rM, cM, rS, lam, mu = mat_esfera
    cL, cT = np.sqrt((lam + 2 * mu) / rS), np.sqrt(mu / rS)
    zL, zT = (cM / cL) * xM, (cM / cT) * xM
    L = ell * (ell + 1)
    with np.errstate(all="ignore"):
        jL, djL, d2jL = J(ell, zL), dJ(ell, zL), d2J(ell, zL)
        jT, djT, d2jT = J(ell, zT), dJ(ell, zT), d2J(ell, zT)
        h, dh, djM = H(ell, xM), dH(ell, xM), dJ(ell, xM)
        PM = rM * cM ** 2 * xM ** 2
        R = 2 * (jL - zL * djL) / (zT ** 2 * d2jT + (L - 2) * jT) if ell > 0 else 0 * zL           # Eq. (5.8)
        AmAt = ((lam * zL ** 2 * jL - 2 * mu * zL ** 2 * d2jL) * dh - rM * cM ** 2 * xM * zL * djL * h) / (PM * h * dh)
        BmBt = (-2 * mu * L * (zT * djT - jT) * dh - rM * cM ** 2 * xM * L * jT * h) / (PM * h * dh)
        CtmC = 1j / (xM ** 2 * h * dh)
        a = CtmC / (AmAt + BmBt * R)                                                                 # Eq. (5.7a), com B~ - B = -(B - B~)
        At, Bt, Ct = zL * djL / (xM * dh), L * jT / (xM * dh), -djM / dh
        return a, R * a, Ct + (At + Bt * R) * a


def energia_esfera_viscoelastica(xM, mat_esfera, lmax=LMAX):
    # Eq. (5.58), normalizada por Z_S = rho_S |c_L|; devolvida em relação a Z_M
    rM, cM, rS, lam, mu = mat_esfera
    cL, cT = np.sqrt((lam + 2 * mu) / rS), np.sqrt(mu / rS)
    xM = np.asarray(xM, float)
    zL, zT = (cM / cL) * xM, (cM / cT) * xM
    zLc, zTc = np.conj(zL), np.conj(zT)
    W = np.zeros_like(xM)
    with np.errstate(all="ignore"):
        for ell in range(lmax + 1):
            a, b, s = esfera_viscoelastica(ell, xM, mat_esfera)
            L, p = ell * (ell + 1), 2 * ell + 1
            if eh_real(zL):        # limite não dissipativo: Lommel (Apêndice H)
                z = np.real(zL)
                FL = z ** 2 * J(ell, z) * dJ(ell, z) + prim_F(ell, z, J, J) / 2
            else:
                FL = np.abs(zL) ** 3 / (zL ** 2 - zLc ** 2) * (zL * J(ell, zL) * dJ(ell, zLc) - zLc * dJ(ell, zL) * J(ell, zLc))
            if eh_real(zT):
                w = np.real(zT)
                FT = np.abs(zL) * J(ell, w) ** 2 + w * np.abs(zL) * J(ell, w) * dJ(ell, w) + np.abs(zL) / (2 * w) * prim_F(ell, w, J, J)
            else:
                FT = (np.abs(zL) * J(ell, zT) * J(ell, zTc)
                      + np.abs(zL) * np.abs(zT) ** 2 / (zT ** 2 - zTc ** 2) * (zT * J(ell, zT) * dJ(ell, zTc) - zTc * dJ(ell, zT) * J(ell, zTc)))
            T = p * np.real(np.abs(a) ** 2 * FL + L * np.abs(b) ** 2 * FT + 2 * L * np.real(a * np.conj(b) * np.abs(zL) * J(ell, zL) * J(ell, zTc)))
            sup = p * (np.conj(a) * dJ(ell, zLc) + np.conj(b) * L / zLc * J(ell, zTc)) * (
                a * zL ** 2 * (lam * J(ell, zL) - 2 * mu * d2J(ell, zL)) - 2 * mu * L * b * (zT * dJ(ell, zT) - J(ell, zT)))
            W += np.nan_to_num(2 * T - np.real(sup / (rS * np.abs(cL) * np.conj(cL))))
    return rS * np.abs(cL) / (rM * cM) * W


# ---------------- conferência dos três limites nos coeficientes ----------------
rM, cM = RHO_AGUA, C_AGUA
x = np.array([0.05, 0.4, 1.2])
print("1. mu -> 0, casca fluida de c_B = 0,3 c_M: comparação com o core-shell fluido em k_M a = x_M/(1 + delta)")
rC, cC, rS, cB, delta = 1024.0, 0.1 * cM, 1100.0, 0.3 * cM, 0.5
rel = lambda u, v: np.max(np.abs(u - v) / np.abs(v))
for razao in (1e-3, 1e-4):
    mu = rS * (razao * cB) ** 2
    mat = materiais(rC, cC, rS, rS * cB ** 2 - 2 * mu / 3, mu)
    for ell in (0, 1, 4):
        a, at, b, bt, c, s = coeficientes(ell, x, delta, mat)
        bf, cf, df, sf, _ = coeficientes_fluido(ell, x / (1 + delta), delta, (rM, rC, rS, cM, cC, cB))
        print(f"   c_T,0/c_B = {razao:.0e}, l = {ell}: |a - c_fl|/|c_fl| = {rel(a, cf):.1e}   |a~ - d_fl|/|d_fl| = {rel(at, df):.1e}   "
              f"|c - b_fl|/|b_fl| = {rel(c, bf):.1e}   |s - s_fl|/|s_fl| = {rel(s, sf):.1e}   |b|/|a| = {np.max(np.abs(b) / np.abs(a)):.1e}")
print("2. delta -> 0, casca de PMMA: comparação com a esfera fluida C de raio b")
mat = materiais(1024.0, 0.2 * cM, 1190.0, *lame_de_velocidades(1190.0, 2700 * np.exp(-1j * np.pi / 180), 1330 * np.exp(-1j * np.pi / 180)))
for d in (1e-5, 1e-7):
    for ell in (0, 1, 4):
        a, at, b, bt, c, s = coeficientes(ell, x, d, mat)
        be, se = esfera(ell, x, (rM, 1024.0, cM, 0.2 * cM))
        print(f"   delta = {d:.0e}, l = {ell}: |c - b_esf|/|b_esf| = {rel(c, be):.1e}   |s - s_esf|/|s_esf| = {rel(s, se):.1e}")
print("3. a -> 0 (a/b = 1e-4), casca de PMMA: comparação com a esfera viscoelástica de raio b")
lam, mu = lame_de_velocidades(1190.0, 2700 * np.exp(-1j * np.pi / 180), 1330 * np.exp(-1j * np.pi / 180))
for ell in (0, 1, 4):
    a, at, b, bt, c, s = coeficientes(ell, x, 1e4 - 1, materiais(1024.0, cM, 1190.0, lam, mu))
    av, bv, sv = esfera_viscoelastica(ell, x, (rM, cM, 1190.0, lam, mu))
    db = f"|b - b_esf|/|b_esf| = {np.max(np.abs(b - bv) / np.abs(bv)):.1e}   " if ell else ""
    print(f"   a/b = 1e-4, l = {ell}: |a - a_esf|/|a_esf| = {np.max(np.abs(a - av) / np.abs(av)):.1e}   {db}"
          f"|s - s_esf|/|s_esf| = {np.max(np.abs(s - sv) / np.abs(sv)):.1e}   |a~|/|a| = {np.max(np.abs(at) / np.abs(a)):.1e}")

In [ ]:
estilo_original()   # rcParams dos notebooks originais (LaTeX, sans-serif)


def salva_fig(fig, base, dpi=600, **kw):
    os.makedirs(os.path.dirname(base), exist_ok=True)
    fig.savefig(base + ".png", dpi=dpi, bbox_inches="tight", **kw)
    fig.savefig(base + ".pdf", bbox_inches="tight", **kw)
    plt.show()
    plt.close(fig)


def virgula_decimal(texto):
    # separador decimal em vírgula (padrão brasileiro); "{,}" evita o espaço extra do LaTeX no modo matemático
    return re.sub(r"(?<=\d)\.(?=\d)", "{,}", str(texto))


def formatador_virgula(ticks):
    # mesmo número de casas decimais para todos os ticks, com vírgula decimal
    casas = max((len(f"{float(t):.10f}".rstrip("0").split(".")[1]) for t in ticks), default=0)
    return FuncFormatter(lambda v, pos: "$" + f"{v:.{casas}f}".replace(".", "{,}") + "$")


FIGSIZE_COMPARE, AXIS_LABELSIZE, TICK_LABELSIZE, LEGEND_FONTSIZE = (8, 6), 18, 15, 12.5
SPINE_WIDTH, TICK_WIDTH, TICK_LENGTH = 1.15, 1.15, 5.0
COLOR_CORE, COLOR_SHELL, COLOR_TOTAL_CS = "#810000", "#0f223d", "0.75"
DIR_LIMITES = os.path.join(FIG_DIR, "limites")
X_MIN_PADRAO, X_MAX_PADRAO = 1e-4, 1.0
XTICKS_XM = [0.25, 0.50, 0.75, 1.00]  # quatro marcas, com vírgula decimal


def plot_limite(xM_vals, curvas, save_name, ylim, save_dir=DIR_LIMITES):
    # mesma estética da Fig. 11: referência em traço-ponto preto, curva convergida em cinza grosso por baixo; ylim[1] = None escolhe o teto
    fig, ax = plt.subplots(figsize=FIGSIZE_COMPARE)
    for c in curvas:
        ax.plot(xM_vals, c["y"], color=c["cor"], linestyle=c["ls"], linewidth=c["lw"], alpha=0.95, zorder=c["z"], label=c["rotulo"])
    ax.set_yscale("log"); ax.set_xlabel(r"$x_M$", fontsize=AXIS_LABELSIZE, labelpad=6)
    ax.set_ylabel(r"Energia interna $(u.a.)$", fontsize=AXIS_LABELSIZE, fontfamily="serif", usetex=USA_LATEX, labelpad=6)
    teto = ylim[1] if ylim[1] is not None else 10.0 ** (np.ceil(np.log10(np.nanmax([np.nanmax(c["y"]) for c in curvas]))) + 1)
    ax.set_xlim(X_MIN_PADRAO, X_MAX_PADRAO); ax.set_ylim(ylim[0], teto); ax.grid(False)
    ax.tick_params(direction="in", length=TICK_LENGTH, width=TICK_WIDTH, labelsize=TICK_LABELSIZE, pad=4)
    for spine in ax.spines.values(): spine.set_linewidth(SPINE_WIDTH)
    ax.xaxis.set_major_locator(FixedLocator(XTICKS_XM))
    ax.xaxis.set_major_formatter(formatador_virgula(XTICKS_XM))
    ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=6)); ax.yaxis.set_minor_formatter(NullFormatter())
    ax.legend(fontsize=LEGEND_FONTSIZE, loc="lower right", framealpha=0.95, edgecolor="black", fancybox=False, borderpad=0.6, labelspacing=0.4)
    fig.subplots_adjust(left=0.18, right=0.97, bottom=0.16, top=0.96)
    salva_fig(fig, os.path.join(save_dir, save_name), dpi=600, pad_inches=0.03)


def refina_pico(f, xs, i):
    # posição (máximo de f entre os vizinhos da malha), altura e largura à meia altura (nan se a meia altura não é alcançada)
    lo, hi = xs[max(i - 1, 0)], xs[min(i + 1, len(xs) - 1)]
    x0 = minimize_scalar(lambda t: -np.log(f(t)), bounds=(lo, hi), method="bounded", options={"xatol": 1e-13 * hi}).x
    W0 = f(x0)

    def meia_altura(sinal):
        # A largura só é definida se a curva cruza W0/2 em ambos os lados do pico.
        alvo = W0 / 2
        if not np.isfinite(alvo) or alvo <= 0:
            return np.nan
        passo, t = max((hi - lo) * 1e-4, 1e-13), x0
        while abs(t - x0) < 0.5 * x0:
            t2 = x0 + sinal * passo
            if not xs[0] < t2 < xs[-1]:
                return np.nan
            g2 = f(t2) - alvo
            if not np.isfinite(g2):
                return np.nan
            if g2 < 0:
                # brentq exige valores de sinais opostos nas extremidades.
                # Tenta primeiro o intervalo local e, se necessário, parte do máximo.
                g1 = f(t) - alvo
                if not np.isfinite(g1) or g1 <= 0:
                    t = x0
                    g1 = f(t) - alvo
                if not np.isfinite(g1) or g1 <= 0:
                    return np.nan
                a, b = sorted((t, t2))
                try:
                    return brentq(lambda u: f(u) - alvo, a, b, xtol=1e-15)
                except ValueError:
                    # Instabilidades numéricas próximas a picos muito estreitos.
                    return np.nan
            t, passo = t2, passo * 1.6
        return np.nan

    return x0, W0, meia_altura(+1) - meia_altura(-1)


def ressonancias(f, xs, W, prominencia=0.1):
    # máximos de log10 W com proeminência >= 0,1 (critério do banco de dados do core-shell fluido), refinados
    idx, _ = find_peaks(np.log10(W), prominence=prominencia)
    return [refina_pico(f, xs, i) for i in idx]


def malha_refinada(x, funcoes, n=121, largura=6.0):
    # acrescenta pontos em torno dos picos de cada curva, para que os picos estreitos apareçam inteiros em todas elas
    extra = []
    for F in funcoes:
        for x0, W0, fw in ressonancias(lambda t: float(F(np.array([t]))[0]), x, F(x)):
            extra.append(x0 + np.linspace(-largura, largura, n) * (fw if np.isfinite(fw) else np.min(np.diff(x))))
    xn = np.unique(np.concatenate([x] + extra))
    return xn[(xn >= x.min()) & (xn <= x.max())]


def desvio(W, W_ref, xM_vals):
    # desvio relativo ponderado pelo passo da malha (a malha refinada concentra pontos nos picos)
    e, w = np.abs(W - W_ref) / np.abs(W_ref), np.gradient(xM_vals)
    ordem = np.argsort(e)
    acum = np.cumsum(w[ordem]) / np.sum(w)
    q = lambda p: e[ordem][np.searchsorted(acum, p)]
    return f"desvio relativo mediano {q(0.5):.1e} (percentil 95: {q(0.95):.1e})"


estilo_tese()

### 4.1 Casca sem rigidez ao cisalhamento: core-shell fluido-fluido

Núcleo lento, $\rho_C=\rho_M$ e $c_C=0{,}1\,c_M$; casca com $\rho_S=1100$ kg/m$^3$ e $K=\rho_Sc_B^2$, $c_B=0{,}3\,c_M$; $\delta=0{,}5$ e $\tan\delta_T=0{,}02$. A referência é o
core-shell fluido do Cap. 4 com $A=C$ e $B$ de densidade $\rho_S$ e velocidade $c_B$. À medida que $c_{T,0}/c_B$ diminui, as ressonâncias de cisalhamento da casca
(estreitas para $c_{T,0}/c_B=0{,}3$ e $0{,}1$) perdem o acoplamento com o campo longitudinal, e a curva viscoelástica cai sobre a fluida. Nos picos
mais estreitos do core-shell fluido (ordens $\ell$ altas, amortecidas só por radiação), a altura da curva viscoelástica ainda é limitada pela pequena
perda no cisalhamento, que desaparece junto com $\mu$. A malha de cada figura recebe pontos extras em torno dos picos de todas as curvas.

In [ ]:
estilo_original()
rC, cC, rS, cB, delta, tan_T = RHO_AGUA, 0.1 * C_AGUA, 1100.0, 0.3 * C_AGUA, 0.5, 0.02
K = rS * cB ** 2
F_fl = lambda xx: energia_total_fluido(xx / (1 + delta), delta, (RHO_AGUA, rC, rS, C_AGUA, cC, cB))
mat_mu = {}
for razao in (0.3, 0.1, 0.001):
    mu = rS * (razao * cB) ** 2 * (1 - 1j * tan_T)
    mat_mu[razao] = materiais(rC, cC, rS, K - 2 * mu / 3, mu)
x = malha_refinada(np.linspace(X_MIN_PADRAO, X_MAX_PADRAO, 3000), [F_fl] + [lambda xx, m=m: energia_total(xx, delta, m) for m in mat_mu.values()])
W_fl = F_fl(x)
W_mu = {razao: energia_total(x, delta, m) for razao, m in mat_mu.items()}
for razao, W in W_mu.items():
    print(f"c_T,0/c_B = {razao}: {desvio(W, W_fl, x)}")
plot_limite(x, [
    dict(y=W_fl, cor="black", ls="-.", lw=1.75, z=6, rotulo=r"Casca-núcleo fluido (Cap. 4)"),
    dict(y=W_mu[0.001], cor=COLOR_TOTAL_CS, ls="-", lw=4.5, z=2, rotulo=r"Casca viscoelástica, $c_{T,0}/c_B=0{,}001$"),
    dict(y=W_mu[0.1], cor=COLOR_SHELL, ls="-", lw=1.6, z=4, rotulo=r"Casca viscoelástica, $c_{T,0}/c_B=0{,}1$"),
    dict(y=W_mu[0.3], cor=COLOR_CORE, ls="--", lw=1.6, z=5, rotulo=r"Casca viscoelástica, $c_{T,0}/c_B=0{,}3$"),
], "limite_mu_zero_coreshell_fluido", ylim=(1e-7, 1e9))
estilo_tese()

### 4.2 Casca de espessura nula: esfera fluida

Núcleo $\rho_C=\rho_M$ e $c_C=0{,}2\,c_M$; casca de PMMA com perda reduzida, $\rho_S=1190$ kg/m$^3$, $c_L=2700\,e^{-1^\circ\mathrm i}$ m/s e $c_T=1330\,e^{-1^\circ\mathrm i}$ m/s.
A referência é a esfera fluida do Cap. 3, de material $C$ e raio $b$. O módulo volumétrico da casca é cerca de sessenta vezes o do núcleo, e por isso mesmo
$\delta=10^{-2}$ ainda desloca as ressonâncias; com $\delta=10^{-4}$ a curva coincide com a da esfera, e a energia da casca cai na mesma proporção de $\delta$.

In [ ]:
estilo_original()
rC, cC = RHO_AGUA, 0.2 * C_AGUA
mat_pmma = (1190.0, *lame_de_velocidades(1190.0, 2700 * np.exp(-1j * np.pi / 180), 1330 * np.exp(-1j * np.pi / 180)))
F_esf = lambda xx: energia_esfera(xx, (RHO_AGUA, rC, C_AGUA, cC))
x = malha_refinada(np.linspace(X_MIN_PADRAO, X_MAX_PADRAO, 3000), [F_esf] + [lambda xx, d=d: energia_total(xx, d, materiais(rC, cC, *mat_pmma)) for d in (1e-2, 1e-4)])
W_esf = F_esf(x)
W_d = {d: energia_total(x, d, materiais(rC, cC, *mat_pmma), partes=True) for d in (1e-2, 1e-4)}
for d, (W, WC, WS) in W_d.items():
    print(f"delta = {d:g}: {desvio(W, W_esf, x)}; energia da casca / total: no máximo {np.max(WS / W):.1e}")
plot_limite(x, [
    dict(y=W_esf, cor="black", ls="-.", lw=1.75, z=6, rotulo=r"Esfera fluida (Cap. 3)"),
    dict(y=W_d[1e-4][0], cor=COLOR_TOTAL_CS, ls="-", lw=4.5, z=2, rotulo=r"Energia total ($\delta=10^{-4}$)"),
    dict(y=W_d[1e-2][0], cor=COLOR_CORE, ls="-", lw=1.6, z=4, rotulo=r"Energia total ($\delta=10^{-2}$)"),
    dict(y=W_d[1e-2][2], cor=COLOR_SHELL, ls="--", lw=1.6, z=5, rotulo=r"Energia da casca ($\delta=10^{-2}$)"),
    dict(y=W_d[1e-4][2], cor=COLOR_SHELL, ls=":", lw=1.8, z=5, rotulo=r"Energia da casca ($\delta=10^{-4}$)"),
], "limite_delta_zero_esfera_fluida", ylim=(1e-8, None))
estilo_tese()

### 4.3 Núcleo de raio nulo: esfera viscoelástica

Núcleo de água; casca do mesmo PMMA da Seção 4.2. A referência é a esfera viscoelástica maciça do Cap. 5, de raio $b$, calculada pela Eq. (5.58),
com a mesma variável $x_M=k_Mb$. Com $a/b=0{,}5$, o núcleo de água ainda desloca e cria ressonâncias; com $a/b=0{,}05$, o núcleo guarda uma fração
da ordem de $10^{-4}$ da energia e a curva coincide com a da esfera maciça.

In [ ]:
estilo_original()
F_vis = lambda xx: energia_esfera_viscoelastica(xx, (RHO_AGUA, C_AGUA, *mat_pmma))
x = malha_refinada(np.linspace(X_MIN_PADRAO, X_MAX_PADRAO, 3000), [F_vis] + [lambda xx, ab=ab: energia_total(xx, 1 / ab - 1, materiais(RHO_AGUA, C_AGUA, *mat_pmma))
                                                         for ab in (0.5, 0.05)])
W_vis = F_vis(x)
W_a = {ab: energia_total(x, 1 / ab - 1, materiais(RHO_AGUA, C_AGUA, *mat_pmma), partes=True) for ab in (0.5, 0.05)}
for ab, (W, WC, WS) in W_a.items():
    print(f"a/b = {ab:g}: {desvio(W, W_vis, x)}; energia do núcleo / total: no máximo {np.max(WC / W):.1e}")
plot_limite(x, [
    dict(y=W_vis, cor="black", ls="-.", lw=1.75, z=6, rotulo=r"Esfera viscoelástica (Cap. 5)"),
    dict(y=W_a[0.05][0], cor=COLOR_TOTAL_CS, ls="-", lw=4.5, z=2, rotulo=r"Energia total ($a/b=0{,}05$)"),
    dict(y=W_a[0.5][0], cor=COLOR_CORE, ls="-", lw=1.6, z=4, rotulo=r"Energia total ($a/b=0{,}5$)"),
    dict(y=W_a[0.5][1], cor=COLOR_SHELL, ls="--", lw=1.6, z=5, rotulo=r"Energia do núcleo ($a/b=0{,}5$)"),
    dict(y=W_a[0.05][1], cor=COLOR_SHELL, ls=":", lw=1.8, z=5, rotulo=r"Energia do núcleo ($a/b=0{,}05$)"),
], "limite_a_zero_esfera_viscoelastica", ylim=(1e-9, None))
estilo_tese()

### 4.4 Conservação de energia e comparação com a literatura

Os três limites conferem a formulação quando uma das regiões desaparece, mas nenhum deles tem, ao mesmo tempo, casca sólida de espessura finita
e núcleo finito. Para esse caso há dois testes independentes.

**Conservação de energia.** Com $\varphi_M\propto j_\ell+s_\ell h^{(1)}_\ell=\tfrac12\big[h^{(2)}_\ell+(1+2s_\ell)\,h^{(1)}_\ell\big]$, a onda que entra em cada multipolo tem
amplitude $1$ e a que sai, $1+2s_\ell$. Sem perda no material, toda a potência que entra é reirradiada, e $|1+2s_\ell|=1$; com perda, $|1+2s_\ell|<1$.
Um termo errado nas condições de contorno, em geral, quebra essa igualdade.

**Soluções publicadas.** Duas soluções para cascas elásticas em água, digitalizadas das figuras originais por Venås e Jenserud no repositório
`e3Dss` (os arquivos são baixados na primeira execução e guardados em `dados_literatura/`):

* Fender (1972), Figs. 2 e 3: casca de alumínio ($\rho_S=2700$ kg/m$^3$, $c_L=6412$ m/s, $c_T=3043$ m/s) com ar dentro ($\rho_C=1{,}21$ kg/m$^3$,
  $c_C=343$ m/s) e água fora ($\rho_M=1026$ kg/m$^3$, $c_M=1500$ m/s), espessura $0{,}05\,b$ ($a/b=0{,}95$). Grandeza: nível da pressão total na
  superfície externa, $20\log_{10}|p/p_{\mathrm{inc}}|$, contra $k_Mb$, no ponto iluminado e no ponto de sombra. Fender mede $\theta$ a partir do ponto
  iluminado; aqui $\theta$ é medido a partir de $+z$, o sentido de propagação, e o ponto iluminado é $\theta=\pi$.
* Chang e Demkowicz (1994), Figs. 16 e 17: casca fina de aço ($E=200$ GPa, $\nu=0{,}3$, $\rho_S=7800$ kg/m$^3$), $h/R=0{,}01$ com raio médio $R=1$,
  vácuo dentro e água fora ($\rho_M=1000$ kg/m$^3$, $c_M=1460$ m/s). Grandeza: parte real da pressão total na superfície externa contra $\theta$,
  para $k=\omega/c_M=15$ e $20$. O vácuo é representado por um núcleo fluido de densidade desprezível ($10^{-6}$ kg/m$^3$), que anula a tração na
  face interna.

Com $p=\mathrm i\omega\rho_M\varphi_M$, a pressão total na superfície externa, relativa à amplitude incidente, é

$$
\frac{p(b,\theta)}{p_{\mathrm{inc}}}=\sum_{\ell=0}^{\infty}(2\ell+1)\,\mathrm i^\ell\big[j_\ell(x_M)+s_\ell\,h^{(1)}_\ell(x_M)\big]P_\ell(\cos\theta),
$$

somada até $\ell=60$, porque $k_Mb$ chega a $32$.

In [ ]:
import urllib.request

# ---------------- conservação de energia: |1 + 2 s_l| = 1 sem perda ----------------
x = np.linspace(X_MIN_PADRAO, X_MAX_PADRAO, 600)
CASOS_CONSERVACAO = {   # (núcleo rho_C, c_C; casca rho_S, c_L, c_T; delta)
    "casca elástica, núcleo de ar":   (1.2, 343.0, 1100.0, 1000.0, 400.0, 0.5),
    "PMMA, núcleo lento":             (1024.0, 300.0, 1190.0, 2700.0, 1330.0, 0.3),
    "capsídeo do CCMV (Seção 5)":     (RHO_AGUA, C_AGUA, 1350.0, 471.4, 192.4, 0.269),
}
for nome, (rC, cC, rS, cL, cT, delta) in CASOS_CONSERVACAO.items():
    mat = materiais(rC, cC, rS, *lame_de_velocidades(rS, cL, cT))
    desvios = [np.nanmax(np.abs(np.abs(1 + 2 * coeficientes(l, x, delta, mat)[5]) - 1)) for l in (0, 1, 2, 5, 9)]
    print(f"{nome:30s} sem perda, max ||1 + 2 s_l| - 1| para l = 0, 1, 2, 5, 9: " + "  ".join(f"{d:.0e}" for d in desvios))
mat = materiais(1024.0, 300.0, 1190.0, *lame_de_velocidades(1190.0, 2700 * np.exp(-0.05j), 1330 * np.exp(-0.05j)))
for l in (0, 1, 2, 5):
    m = np.abs(1 + 2 * coeficientes(l, x, 0.3, mat)[5])
    print(f"PMMA com perda (fase de 0,05 rad), l = {l}: |1 + 2 s_l| entre {np.nanmin(m):.4f} e {np.nanmax(m):.12f}")

# ---------------- soluções publicadas, digitalizadas no repositório e3Dss ----------------
URL_E3DSS = "https://raw.githubusercontent.com/Zetison/e3Dss/a12c01cc80dfdb1d9f1e9bc5822bbd4e9afd83fe/models/"
PASTA_LIT = os.path.join(os.path.dirname(SAIDA), "dados_literatura")


def dados_literatura(caminho):
    local = os.path.join(PASTA_LIT, caminho)
    if not os.path.exists(local):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        urllib.request.urlretrieve(URL_E3DSS + caminho, local)
    return np.loadtxt(local, delimiter=",")


def pressao_superficie(xM, theta, delta, mat, lmax=60):
    # p(b, theta)/p_inc = soma de (2l+1) i^l [j_l(x_M) + s_l h_l(x_M)] P_l(cos theta); termos não finitos (y_l estourado em argumento pequeno) são desprezíveis
    xM, theta = np.asarray(xM, float), np.asarray(theta, float)
    p = np.zeros(np.broadcast(xM, theta).shape, complex)
    for ell in range(lmax + 1):
        s = coeficientes(ell, xM, delta, mat)[5]
        with np.errstate(all="ignore"):
            termo = (2 * ell + 1) * 1j ** ell * (J(ell, xM) + s * H(ell, xM)) * eval_legendre(ell, np.cos(theta))
        p += np.where(np.isfinite(termo), termo, 0)
    return p


MAT_FENDER = materiais(1.21, 343.0, 2700.0, *lame_de_velocidades(2700.0, 6412.0, 3043.0), rho_M=1026.0, c_M=1500.0)
DELTA_FENDER = 1 / 0.95 - 1
FENDER = {"iluminado": (dados_literatura("Fender1972sfa/Fig2.csv"), np.pi), "de sombra": (dados_literatura("Fender1972sfa/Fig3.csv"), 0.0)}
x_fender = np.linspace(1e-3, 32.0, 8000)
for lado, (ref, th) in FENDER.items():
    SPL = 20 * np.log10(np.abs(pressao_superficie(np.clip(ref[:, 0], 1e-3, None), th, DELTA_FENDER, MAT_FENDER)))
    d = np.abs(SPL - ref[:, 1])
    print(f"Fender (1972), ponto {lado}: |diferença| mediana {np.median(d):.2f} dB, percentil 90 {np.percentile(d, 90):.2f} dB")

R_CHANG, h_CHANG = 1.0, 0.01
MAT_CHANG = materiais(1e-6, 343.0, 7800.0, 2.0e11 * 0.3 / (1.3 * 0.4), 2.0e11 / 2.6, rho_M=1000.0, c_M=1460.0)
DELTA_CHANG = (R_CHANG + h_CHANG / 2) / (R_CHANG - h_CHANG / 2) - 1
CHANG = {15: dados_literatura("Chang1994voa2/Figure16.csv"), 20: dados_literatura("Chang1994voa2/Figure17.csv")}
for k, ref in CHANG.items():
    p = pressao_superficie(k * (R_CHANG + h_CHANG / 2), np.radians(ref[:, 0]), DELTA_CHANG, MAT_CHANG)
    d = np.real(p) - ref[:, 1]
    print(f"Chang e Demkowicz (1994), k = {k}: diferença rms {np.sqrt(np.mean(d ** 2)):.3f}, máxima {np.max(np.abs(d)):.3f} "
          f"(amplitude da curva: {np.ptp(ref[:, 1]):.2f})")

In [ ]:
estilo_original()


def painel(ax, x_ref, y_ref, x_mod, y_mod, rotulo_ref, xlabel, ylabel, texto, xlim, ylim, xt):
    ax.plot(x_ref, y_ref, color=COLOR_TOTAL_CS, linestyle="-", linewidth=4.0, zorder=2, label=rotulo_ref)
    ax.plot(x_mod, y_mod, color=COLOR_SHELL, linestyle="-", linewidth=1.0, zorder=4, label=r"Casca-núcleo viscoelástico")
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.grid(False)
    ax.set_xlabel(xlabel, fontsize=AXIS_LABELSIZE, labelpad=6)
    ax.set_ylabel(ylabel, fontsize=AXIS_LABELSIZE - 2, labelpad=6)
    ax.tick_params(direction="in", length=TICK_LENGTH, width=TICK_WIDTH, labelsize=TICK_LABELSIZE - 2, pad=4)
    for spine in ax.spines.values(): spine.set_linewidth(SPINE_WIDTH)
    ax.xaxis.set_major_locator(FixedLocator(xt)); ax.xaxis.set_major_formatter(formatador_virgula(xt))
    yt = [t for t in ax.get_yticks() if ylim[0] <= t <= ylim[1]]
    ax.yaxis.set_major_locator(FixedLocator(yt)); ax.yaxis.set_major_formatter(formatador_virgula(yt))
    ax.text(0.02, 0.96, texto, transform=ax.transAxes, ha="left", va="top", fontsize=LEGEND_FONTSIZE, zorder=10,
            bbox=dict(boxstyle="square,pad=0.35", facecolor="white", edgecolor="black", linewidth=1.0))
    ax.legend(fontsize=LEGEND_FONTSIZE - 1, loc="upper right", framealpha=0.95, edgecolor="black", fancybox=False, borderpad=0.5, labelspacing=0.35)


fig, axs = plt.subplots(2, 1, figsize=(10, 9))
for ax, (lado, (ref, th)), letra in zip(axs, FENDER.items(), "ab"):
    SPL = 20 * np.log10(np.abs(pressao_superficie(x_fender, th, DELTA_FENDER, MAT_FENDER)))
    painel(ax, ref[:, 0], ref[:, 1], x_fender, SPL, r"Fender (1972)", r"$k_Mb$", r"$20\log_{10}|p/p_{\mathrm{inc}}|$ (dB)",
           f"[{letra}] ponto {lado}", (0, 32), (-30, 50), [0, 4, 8, 12, 16, 20, 24, 28, 32])
fig.subplots_adjust(left=0.11, right=0.98, bottom=0.08, top=0.98, hspace=0.32)
salva_fig(fig, os.path.join(DIR_LIMITES, "literatura_fender1972"), dpi=600, pad_inches=0.03)

fig, axs = plt.subplots(2, 1, figsize=(10, 8))
theta_g = np.linspace(0, 180, 1801)
for ax, (k, ref), letra in zip(axs, CHANG.items(), "ab"):
    p = pressao_superficie(k * (R_CHANG + h_CHANG / 2), np.radians(theta_g), DELTA_CHANG, MAT_CHANG)
    painel(ax, ref[:, 0], ref[:, 1], theta_g, np.real(p), r"Chang e Demkowicz (1994)", r"$\theta$ (graus)",
           r"$\mathrm{Re}\,(p/p_{\mathrm{inc}})$", f"[{letra}] $k={k}$, $h/R=0{{,}}01$", (0, 180), (-2, 2.4), [0, 45, 90, 135, 180])
fig.subplots_adjust(left=0.11, right=0.98, bottom=0.09, top=0.98, hspace=0.32)
salva_fig(fig, os.path.join(DIR_LIMITES, "literatura_chang1994"), dpi=600, pad_inches=0.03)
estilo_tese()

Nas duas comparações, o modelo reproduz as soluções publicadas. Para a casca de Fender, as posições das ressonâncias e o nível entre elas coincidem
com a curva digitalizada (diferença mediana de $0{,}2$ dB no ponto iluminado e de $0{,}4$ dB no de sombra); as diferenças maiores ficam no topo dos
picos e nos picos estreitíssimos que só aparecem no modelo, ressonâncias de $Q$ alto que a malha de frequências de 1972 e a espessura do traço da
figura não resolvem. Para a casca de Chang e Demkowicz, ainda mais fina que as dos capsídeos da Seção 5 ($h/R=0{,}01$, contra $0{,}06$ a $0{,}24$), a diferença rms é
de cerca de $1\,\%$ da amplitude da curva, o nível da própria digitalização. Junto com a conservação de energia, isso confere as condições de contorno e os
coeficientes no caso de casca sólida finita, que os limites não alcançam.

## 5. Vírus da literatura: capsídeos como cascas finas

Em capsídeos virais, a casca proteica tem poucos nanômetros de espessura diante de raios de 10 a 50 nm, e os ensaios de nanoindentação são
interpretados pela teoria de cascas finas, que fornece um módulo de Young efetivo $E$ associado à espessura $h$. Esse $E$ só tem sentido junto com
$h$: aplicado a uma esfera maciça do raio do vírus, ele descreve outro objeto. A geometria do core-shell viscoelástico é a mesma da teoria de
cascas usada nesses ajustes, e por isso os parâmetros podem ser usados diretamente.

A Tabela 1 de Roos, Bruinsma e Wuite (2010) reúne raio médio $R$ (média dos raios interno e externo), espessura $h$ e $E$ para sete capsídeos
aproximadamente esféricos (o pró-capsídeo prolato do $\phi29$ é tratado como esfera, como na própria tabela; quando a tabela traz dois valores de $E$,
usa-se o primeiro). Aqui, $a=R-h/2$, $b=R+h/2$ e $\delta=h/(R-h/2)$. As demais hipóteses são:

* casca: $\nu=0{,}4$ (o valor usado na tabela) e $\rho_S=1350$ kg/m$^3$, densidade típica de proteínas; dissipação só no cisalhamento (Seção 2).
  Sem perda ($\tan\delta_T=0$), as ressonâncias da casca só perdem energia por radiação e são estreitíssimas, com $Q$ de $5\times10^4$ a $5\times10^6$; por isso as
  figuras mostram $\tan\delta_T=10^{-3}$ (quase elástica) e $\tan\delta_T=0{,}02$, valor ilustrativo, e a tabela traz o $Q$ sem perda e o $Q$ com $\tan\delta_T=0{,}02$;
* núcleo e meio externo: água ($\rho=1024$ kg/m$^3$, $c=1522$ m/s), como em capsídeos vazios; com o genoma empacotado, o núcleo seria mais denso;
* comparação: a esfera viscoelástica maciça do Cap. 5, do mesmo material e raio $b$, que é o modelo de que se dispõe sem a casca.

A frequência é $f=x_Mc_M/(2\pi b)$ (eixo superior). Os picos são localizados como no banco de dados do core-shell fluido (máximos de
$\log_{10}\widetilde W_T$ com proeminência de pelo menos $0{,}1$); a posição é refinada maximizando $\widetilde W_T$ entre os pontos vizinhos, a largura à
meia altura é obtida por busca de raiz, $Q=x_{M}/\mathrm{FWHM}$, e a ordem $\ell$ dominante é a de maior energia modal no pico. A malha das figuras
recebe pontos extras em torno de cada pico, para que a linha alcance o marcador.

In [ ]:
estilo_original()
from matplotlib.ticker import MaxNLocator

# Roos, Bruinsma & Wuite (2010), Nature Physics 6, 733, Tabela 1: R médio (nm), h (nm), E (GPa)
VIRUS = {
    r"$\phi29$":  dict(R=23.2, h=1.6, E=1.80, arq="phi29"),
    r"$\lambda$": dict(R=29.5, h=1.8, E=1.00, arq="lambda"),
    "HSV-1":      dict(R=49.5, h=4.0, E=1.00, arq="hsv1"),
    "MVM":        dict(R=11.5, h=2.0, E=1.25, arq="mvm"),
    "CCMV":       dict(R=11.8, h=2.8, E=0.14, arq="ccmv"),
    "HBV T3":     dict(R=11.9, h=2.4, E=0.37, arq="hbv_t3"),
    "HBV T4":     dict(R=13.6, h=2.1, E=0.36, arq="hbv_t4"),
}
NU_CAPSIDEO, RHO_CAPSIDEO = 0.4, 1350.0
TAN_QUASE_ELASTICA, TAN_PERDA = 1e-3, 0.02          # curvas da figura; Q sem perda vem de tan delta_T = 0
X_MAX_VIRUS = X_MAX_PADRAO
DIR_VIRUS = os.path.join(FIG_DIR, "virus")
x_base = np.unique(np.concatenate([np.geomspace(X_MIN_PADRAO, 0.1, 1000), np.linspace(0.1, X_MAX_VIRUS, 3000)]))


def l_dominante(x0, delta, mat):
    return int(np.argmax([energia_modal(l, x0, delta, mat)[0] for l in range(LMAX + 1)]))


RES_VIRUS = {}
for nome, v in VIRUS.items():
    t0 = time.time()
    a, b = v["R"] - v["h"] / 2, v["R"] + v["h"] / 2
    delta = b / a - 1
    mat = {tg: materiais(RHO_AGUA, C_AGUA, RHO_CAPSIDEO, *lame_de_E_nu(v["E"] * 1e9, NU_CAPSIDEO, tg)) for tg in (0.0, TAN_QUASE_ELASTICA, TAN_PERDA)}
    mat_esf = (RHO_AGUA, C_AGUA, RHO_CAPSIDEO, *lame_de_E_nu(v["E"] * 1e9, NU_CAPSIDEO, TAN_QUASE_ELASTICA))
    F = {tg: (lambda xx, m=m: energia_total(xx, delta, m)) for tg, m in mat.items()}
    F_esf = lambda xx: energia_esfera_viscoelastica(xx, mat_esf)
    escalar = lambda G: (lambda t: float(G(np.array([t]))[0]))
    # primeira ressonância: posição e Q sem perda (só radiação) e com tan delta_T = TAN_PERDA; esfera maciça
    W0_base = F[0.0](x_base)
    i0 = find_peaks(np.log10(W0_base), prominence=0.1)[0][0]
    x1, W1, fw1 = refina_pico(escalar(F[0.0]), x_base, i0)
    picos_pd = ressonancias(escalar(F[TAN_PERDA]), x_base, F[TAN_PERDA](x_base))
    picos_esf = ressonancias(escalar(F_esf), x_base, F_esf(x_base))
    # curvas da figura, com pontos extras em torno dos picos da curva quase elástica
    x_fig = malha_refinada(x_base, [F[TAN_QUASE_ELASTICA]])
    W_qe = F[TAN_QUASE_ELASTICA](x_fig)
    picos_qe = ressonancias(escalar(F[TAN_QUASE_ELASTICA]), x_fig, W_qe)
    RES_VIRUS[nome] = dict(a=a, b=b, delta=delta, x_fig=x_fig, W_qe=W_qe, W_pd=F[TAN_PERDA](x_fig), W_esf=F_esf(x_fig),
                           primeira=(x1, W1, fw1), l1=l_dominante(x1, delta, mat[0.0]), picos_qe=picos_qe, picos_pd=picos_pd, picos_esf=picos_esf,
                           GHz=C_AGUA / (2 * np.pi * b * 1e-9) / 1e9, cL=abs(velocidades(mat[0.0])[0]), cT=abs(velocidades(mat[0.0])[1]))
    r = RES_VIRUS[nome]
    print(f"{v['arq']:7s} delta = {delta:.3f} | c_L = {r['cL']:.0f} m/s, c_T = {r['cT']:.0f} m/s | primeira ressonância: x_M = {x1:.4f} "
          f"(l = {r['l1']}), f = {x1 * r['GHz']:.2f} GHz | picos até x_M = {X_MAX_VIRUS}: {len(picos_qe)} (tan d_T = {TAN_QUASE_ELASTICA}), "
          f"{len(picos_pd)} (tan d_T = {TAN_PERDA}), {len(picos_esf)} (esfera maciça) | {time.time() - t0:.1f} s")


def plot_virus(nome, v, r):
    fig, ax = plt.subplots(figsize=(6.5, 5.0))
    ax.plot(r["x_fig"], r["W_qe"], color="#0f223d", linestyle="-", linewidth=1.4, zorder=3,
            label=r"Casca-núcleo, $\tan\delta_T=10^{-3}$")
    ax.plot(r["x_fig"], r["W_pd"], color="#810000", linestyle="--", linewidth=1.8, zorder=4,
            label=r"Casca-núcleo, $\tan\delta_T=" + dec(TAN_PERDA, 2) + "$")
    ax.plot(r["x_fig"], r["W_esf"], color="black", linestyle="-.", linewidth=1.3, zorder=2, label=r"Esfera maciça (Cap. 5)")
    if r["picos_qe"]:
        ax.scatter([p[0] for p in r["picos_qe"]], [p[1] for p in r["picos_qe"]], s=22, color="#0f223d", edgecolors="white", linewidths=0.45, zorder=5)
    ax.set_yscale("log")
    ax.set_xlim(X_MIN_PADRAO, X_MAX_VIRUS)
    curvas = np.concatenate([r["W_qe"], r["W_pd"], r["W_esf"]])
    piso = 10.0 ** np.floor(np.log10(np.percentile(r["W_qe"], 2)))
    teto = 10.0 ** (np.ceil(np.log10(np.max(curvas))) + 3)            # espaço para a caixa de parâmetros
    ax.set_ylim(piso, teto)
    ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=6)); ax.yaxis.set_minor_formatter(NullFormatter()); ax.minorticks_off()
    ax.xaxis.set_major_locator(FixedLocator(XTICKS_XM))
    ax.xaxis.set_major_formatter(formatador_virgula(XTICKS_XM))
    ax.set_xlabel(r"$x_M=k_Mb$", fontsize=18, labelpad=6)
    ax.set_ylabel(r"Energia interna $(u.a.)$", fontsize=18, fontfamily="serif", usetex=USA_LATEX, labelpad=6)
    ax.tick_params(direction="in", length=5, width=1.2, labelsize=14, pad=5)
    topo = ax.secondary_xaxis("top", functions=(lambda t: t * r["GHz"], lambda f: f / r["GHz"]))
    topo.set_xlabel(r"$f$ (GHz)", fontsize=15, labelpad=6)
    topo.tick_params(direction="in", length=4, width=1.0, labelsize=12)
    topo.xaxis.set_major_locator(FixedLocator([t * r["GHz"] for t in XTICKS_XM]))
    topo.xaxis.set_major_formatter(FuncFormatter(lambda t, pos: "$" + virgula_decimal(f"{t:g}") + "$"))
    for spine in ax.spines.values():
        spine.set_linewidth(1.5); spine.set_edgecolor("black")
    ax.grid(False)
    caixa = (nome + "\n" + r"$R=" + virgula_decimal(f"{v['R']:g}") + r"$ nm, $h=" + virgula_decimal(f"{v['h']:g}") + r"$ nm" + "\n"
             + r"$E=" + virgula_decimal(f"{v['E']:g}") + r"$ GPa, $\delta=" + dec(r["delta"], 3) + "$")
    ax.text(0.035, 0.955, caixa, transform=ax.transAxes, ha="left", va="top", fontsize=13, linespacing=1.35, zorder=10,
            bbox={"boxstyle": "square,pad=0.35", "facecolor": "white", "edgecolor": "black", "linewidth": 1.5, "alpha": 1.0})
    leg = ax.legend(loc="lower right", fontsize=11, frameon=True, handlelength=2.6, borderpad=0.5, labelspacing=0.35, fancybox=False)
    leg.get_frame().set(edgecolor="black", facecolor="white", linewidth=1.2, alpha=1.0)
    fig.subplots_adjust(left=0.17, right=0.97, bottom=0.15, top=0.86)
    salva_fig(fig, os.path.join(DIR_VIRUS, "energia_" + v["arq"]), dpi=600, pad_inches=0.05)


for nome, v in VIRUS.items():
    plot_virus(nome, v, RES_VIRUS[nome])
estilo_tese()

### Tabela da primeira ressonância

Para cada capsídeo: a ordem $\ell$ dominante, a posição $x_{M,1}$ e a frequência $f_1$ da primeira ressonância do core-shell sem perda, o fator de qualidade
sem perda (só radiação) e com $\tan\delta_T=0{,}02$, e a primeira ressonância da esfera maciça do mesmo material e raio ($\tan\delta_T=10^{-3}$). Mesmo formato das tabelas da tese,
em PNG, PDF, SVG, CSV e `.tex`.

In [ ]:
def save_compact_table(base, dados, linhas, colunas, figsize, first_col_width=0.16, header_label="Coef.", fontsize=17):
    with plt.rc_context({"font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 30}):
        fig, ax = plt.subplots(figsize=figsize, facecolor="white")
        ax.axis("off")
        tab = ax.table(cellText=[[linhas[i], *dados[i]] for i in range(len(linhas))], colLabels=[header_label, *colunas],
                       cellLoc="center", colWidths=[first_col_width] + [(1 - first_col_width) / len(colunas)] * len(colunas),
                       bbox=[0.02, 0.02, 0.96, 0.96])
        tab.auto_set_font_size(False)
        tab.set_fontsize(fontsize)
        for (lin, col), cel in tab.get_celld().items():
            cel.visible_edges = ""
            cel.set_linewidth(0.0)
            cel.PAD = 0.01
            cel.get_text().set_ha("left" if col == 0 else "center")
            cel.get_text().set_linespacing(0.90)
        for col in range(len(colunas) + 1):
            tab[(0, col)].visible_edges = "TB"
            tab[(0, col)].set_linewidth(1.5)
            tab[(0, col)].get_text().set_fontsize(fontsize + 1)
            tab[(len(linhas), col)].visible_edges = "B"
            tab[(len(linhas), col)].set_linewidth(1.5)
        tab[(0, 0)].get_text().set_fontweight("bold")
        fig.subplots_adjust(left=0.005, right=0.995, bottom=0.005, top=0.995)
        for ext in ("png", "pdf", "svg"):
            fig.savefig(f"{base}.{ext}", dpi=600, bbox_inches="tight", pad_inches=0.01, facecolor="white")
        plt.show()
        plt.close(fig)


def salvar_tabela(nome, linhas, colunas, dados, figsize, first_col_width=0.16, fontsize=17, header_label="Coef.", csv=None):
    # figura da tabela + .tex (as células já vêm formatadas, em modo matemático) + .csv com os números
    base = os.path.join(TAB_DIR, nome)
    save_compact_table(base, dados, linhas, colunas, figsize, first_col_width, header_label, fontsize)
    with open(base + ".tex", "w", encoding="utf-8") as fh:
        fh.write("\\begin{tabular}{l" + "c" * len(colunas) + "}\n\\hline\n")
        fh.write(header_label + " & " + " & ".join(c.replace("\n", " ") for c in colunas) + " \\\\\n\\hline\n")
        for i, l in enumerate(linhas):
            fh.write(l + " & " + " & ".join(dados[i]) + " \\\\\n")
        fh.write("\\hline\n\\end{tabular}\n")
    if csv is not None:
        csv.to_csv(base + ".csv", index=False)


def cientifico(v, casas=1):
    if not np.isfinite(v):
        return "--"
    e = int(np.floor(np.log10(abs(v))))
    return rf"${dec(v / 10 ** e, casas)}\times10^{{{e}}}$" if abs(e) >= 3 else f"${dec(v, 1)}$"


linhas, dados, registros = [], [], []
for nome, v in VIRUS.items():
    r = RES_VIRUS[nome]
    x1, W1, fw1 = r["primeira"]
    xp, Wp, fwp = r["picos_pd"][0] if r["picos_pd"] else (np.nan,) * 3
    xe, We, fwe = r["picos_esf"][0] if r["picos_esf"] else (np.nan,) * 3
    linhas.append(nome)
    dados.append([f"${dec(r['delta'], 3)}$", f"${r['l1']}$", f"${dec(x1, 4)}$", f"${dec(x1 * r['GHz'], 2)}$", cientifico(x1 / fw1),
                  f"${dec(xp / fwp, 1)}$" if np.isfinite(fwp) else "--", f"${dec(xe, 4)}$", f"${dec(xe * r['GHz'], 2)}$"])
    registros.append(dict(virus=v["arq"], R_nm=v["R"], h_nm=v["h"], E_GPa=v["E"], a_nm=r["a"], b_nm=r["b"], delta=r["delta"], l_dominante=r["l1"],
                          xM1=x1, f1_GHz=x1 * r["GHz"], Q_elastico=x1 / fw1, xM1_perda=xp, Q_perda=xp / fwp,
                          xM1_esfera_macica=xe, f1_esfera_macica_GHz=xe * r["GHz"], razao_f_macica_coreshell=xe / x1))
colunas = [r"$\delta$", r"$\ell$", r"$x_{M,1}$", r"$f_1$ (GHz)", r"$Q$, $\tan\delta_T=0$", r"$Q$, $\tan\delta_T=0{,}02$",
           r"$x_{M,1}$ maciça", r"$f_1$ maciça (GHz)"]
TABELA_VIRUS = pd.DataFrame(registros)
salvar_tabela("tabela_virus_primeira_ressonancia", linhas, colunas, dados, figsize=(15.5, 4.6), first_col_width=0.10, fontsize=15,
              header_label="Vírus", csv=TABELA_VIRUS)
pd.set_option("display.width", 200)
print(TABELA_VIRUS[["virus", "delta", "l_dominante", "xM1", "f1_GHz", "Q_elastico", "Q_perda", "xM1_esfera_macica", "razao_f_macica_coreshell"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4g}"))

### Discussão

Nos sete capsídeos, a primeira ressonância do core-shell é quadrupolar ($\ell=2$) e fica em $0{,}11\lesssim x_{M,1}\lesssim0{,}27$, entre cerca de
$0{,}8$ e $5{,}3$ GHz. Nesse modo a casca se deforma sem variar o volume interno, de modo que o núcleo de água praticamente não é comprimido; a
frequência resulta da rigidez da casca contra a inércia da própria casca e do fluido que ela desloca. Sem perda no material, só a radiação amortece
o modo, e o fator de qualidade vai de $5\times10^4$ a $5\times10^6$; com $\tan\delta_T=0{,}02$, cai para cerca de $50$ (cerca de $25$ no CCMV). A largura das
ressonâncias dos capsídeos é, portanto, ditada pela dissipação da proteína, que é o parâmetro menos conhecido do problema.

A esfera maciça do mesmo material e raio tem a primeira ressonância entre $2{,}5$ e $4{,}9$ vezes mais alta. O modo de mais baixa frequência do
capsídeo é um modo de casca, que a esfera maciça não possui; por isso, usar o $E$ de nanoindentação em um modelo de esfera homogênea desloca a
primeira ressonância por um fator de $2{,}5$ a $5$, e a geometria de casca fina é indispensável para os vírus.

## 6. Arquivos gerados

Lista o que foi gravado em `resultados/` e cria `resultados_coreshell_viscoelastico.zip` ao lado da pasta (no Colab, dentro de
`Meu Drive/coreshell_viscoelastico/`), pronto para baixar.

In [ ]:
for sub in (FIG_DIR, TAB_DIR):
    arqs = sorted(os.path.relpath(os.path.join(raiz, a), sub) for raiz, _, nomes in os.walk(sub) for a in nomes)
    itens = sorted({os.path.splitext(a)[0] for a in arqs})
    print(f"{sub}: {len(arqs)} arquivos, {len(itens)} itens")
    for n in itens:
        print("   ", n)
zip_path = shutil.make_archive(os.path.join(os.path.dirname(SAIDA), "resultados_coreshell_viscoelastico"), "zip",
                               root_dir=os.path.dirname(SAIDA), base_dir=os.path.basename(SAIDA))
print("zip:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
if NO_COLAB:
    print("No Colab os arquivos já estão no Drive; a sincronização pode levar alguns minutos para aparecer no navegador.")